# ECMC Pipeline Failure Risk Model — Corrected Methodology (Honest Train/Validation/Test Split)

This notebook runs the corrected training pipeline in
[`Code/train_pipeline_risk_honest_split.py`](Code/train_pipeline_risk_honest_split.py), which
fixes a methodological issue identified during SPE Journal peer review: the original script
selected the best model, derived the T2/T3 risk thresholds, and decided whether calibration
helped — all using the same 30% "test" split that was then reused to report headline
performance numbers (including the paper's "83.9% of confirmed failures fall in High/Severe"
claim). Three reviewers independently caught variants of this issue.

**What changed**: a proper three-way split (**60% train / 20% validation / 20% test**) is used
instead of a single 70/30 split.
- **Train** fits the models (with SMOTE/TomekLinks applied only inside CV folds, as before).
- **Validation** selects the best model, derives the T2/T3 thresholds, and decides whether
  probability calibration helps. None of this touches the test set.
- **Test** is referenced exactly once, in **Step 18**, purely to report final, honest
  generalisation numbers.

**Two kinds of numbers now coexist, clearly labeled apart:**
- **Operational registry scoring** (Steps 13-16): every one of the 15,331 real pipelines —
  including ones the model trained on — gets scored, exactly as a deployed risk-scoring tool
  would. This is legitimate for producing the actual risk registry, but it is **not** a
  generalisation metric.
- **Honest held-out evaluation** (Step 18): the test set's own scores, under thresholds it was
  never used to derive. This is what should be cited in the paper as the model's true
  discriminative ability.

**Also includes**: a Brier score reliability diagnostic (in addition to PR-AUC) for the
calibration decision, directly requested by reviewers alongside/instead of a reliability
diagram.

Every code cell below is a verbatim slice of `train_pipeline_risk_honest_split.py` — the only
edits are the four path constants at the top (pointed at this repo's local `Data/` and a fresh
`Results_Honest/` output folder) and the addition of the final two sections (follow-up
importance/SHAP/ablation analysis, and the interactive risk-assessment tool).


## Step 0 — Configuration, imports, plotting style

Only the four path constants differ from the original script.

In [1]:
import os
# =============================================================================
# Pipeline Risk Classification — Binary Training → Continuous Score → 3 Classes
# Based on: "Machine Learning Models for Predicting Pipeline Failures"
#
# PHASE 1 — BINARY TRAINING (clean ground truth)
#   High (1) → failure_count >= 1  (608 confirmed ECMC spill incidents)
#   Low  (0) → failure_count == 0  (14,723 no recorded incident)
#   All 11 features used. No leakage. Every metric is honest.
#
# PHASE 2 — CONTINUOUS RISK SCORE
#   Best model outputs pred_prob_high ∈ [0,1] for every pipeline.
#   This score reflects how much each pipeline resembles historical failures.
#
# PHASE 3 — POST-TRAINING THREE-CLASS ASSIGNMENT
#   Score distribution fitted with Gaussian KDE.
#   Thresholds T1, T2 derived from distribution statistics.
#   Low    → score < T1
#   Medium → T1 <= score < T2   (grey zone — shares failure characteristics)
#   High   → score >= T2
#   Thresholds can be adjusted anytime without retraining.
# =============================================================================
# ─────────────────────────────────────────────────────────────────────────────
#  CONFIGURE PATHS HERE
# ─────────────────────────────────────────────────────────────────────────────

DATA_PATH    = "Data/pipeline_level_dataset__2024.csv"
RESULTS_DIR  = "Results_Honest"
MODELS_DIR   = "Results_Honest/Models"
PLOTS_DIR    = "Results_Honest/Plots"

RAND_SEED = 42

# Post-training thresholds — adjust freely without retraining
# Four risk classes: Low | Medium | High | Severe
#
#   T1 — Low | Medium boundary
#        Pipelines below T1 look like the safe population. No action needed.
#        Default: 0.10  (above safe population mean of ~0.04)
#
#   T2 — Medium | High boundary
#        Pipelines above T2 have a clear failure signal. Schedule inspection.
#        Set at F1-optimal threshold from PR curve (~0.30) — operationally
#        meaningful: flag if model assigns ≥30% failure probability.
#
#   T3 — High | Severe boundary
#        Pipelines above T3 are those the model is most confident about.
#        These are the PR-optimal threshold from the test set (~0.93).
#        Immediate action — do not wait for scheduled inspection cycle.
#
# Override any of these freely without retraining:
T1_OVERRIDE = 0.05 # Low|Medium   — below safe population
T2_OVERRIDE = 0.30   # Medium|High  — F1-optimal, ~30% failure probability
T3_OVERRIDE = None   # High|Severe  — auto = PR-optimal threshold from test set

# ─────────────────────────────────────────────────────────────────────────────

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy import stats
import joblib

warnings.filterwarnings('ignore')
np.random.seed(RAND_SEED)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.pipeline        import Pipeline
from sklearn.impute          import SimpleImputer
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import (RandomForestClassifier,
                                     GradientBoostingClassifier,
                                     StackingClassifier)
from sklearn.svm             import SVC
from sklearn.metrics         import (classification_report, confusion_matrix,
                                     ConfusionMatrixDisplay, roc_curve, auc,
                                     accuracy_score, roc_auc_score,
                                     balanced_accuracy_score, f1_score,
                                     precision_recall_curve,
                                     average_precision_score,
                                     brier_score_loss)
from imblearn.over_sampling  import BorderlineSMOTE
from imblearn.under_sampling import TomekLinks
from imblearn.pipeline       import Pipeline as ImbPipeline
from xgboost                 import XGBClassifier

for d in [RESULTS_DIR, MODELS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"Ready: {d}")

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 15,
    'font.weight': 'normal',
    'axes.linewidth': 1.0,
    'axes.labelsize': 16,
    'axes.labelweight': 'normal',
    'axes.titlesize': 18,
    'axes.titleweight': 'normal',
    'xtick.labelsize': 14,
    'ytick.labelsize': 14,
    'legend.frameon': False,
    'legend.fontsize': 13,
    'figure.dpi': 300,
    'savefig.dpi': 600,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

# A4-friendly shared type scale for all generated figures.
FS_TITLE    = 18
FS_SUPTITLE = 20
FS_LABEL    = 16
FS_TICK     = 14
FS_LEGEND   = 13
FS_ANNOT    = 14
FS_NUMBER   = 16
FS_NOTE     = 12

def threshold_label_position(x, x_max=1.0, pad=0.012, right_margin=0.14):
    """Keep threshold labels inside the plotting area near axis edges."""
    if x >= x_max - right_margin:
        return x - pad, 'right'
    return x + pad, 'left'

def add_bottom_note(fig, text, y=0.012):
    fig.text(0.5, y, text, ha='center', va='bottom',
             fontsize=FS_NOTE, color='#2F2F2F')

# Bright, colorblind-aware palette used consistently across figures.
COLOR_SAFE     = '#0072B2'   # bright blue
COLOR_FAIL     = '#D55E00'   # bright vermillion
COLOR_OVERLAP  = '#CC79A7'   # bright purple
COLOR_MEDIUM   = '#E69F00'   # bright amber
COLOR_SUCCESS  = '#009E73'   # bright green
COLOR_HIGH     = COLOR_FAIL
COLOR_SEVERE   = '#CC3311'   # deep bright red-orange
COLOR_NEUTRAL  = '#2F2F2F'

TEXT_SAFE      = '#004C73'
TEXT_MEDIUM    = '#7A4A00'
TEXT_FAIL      = '#7A2E00'
TEXT_SEVERE    = '#7A1F11'
TEXT_SUCCESS   = '#005F46'

ANNOT_BOX = dict(boxstyle='round,pad=0.18', facecolor='white',
                 edgecolor='#777777', alpha=0.92)

RISK_COLORS = {'Severe': COLOR_SEVERE, 'High': COLOR_HIGH,
               'Medium': COLOR_MEDIUM, 'Low': COLOR_SAFE}
RISK_LS     = {'Severe': '-',          'High': '-',
               'Medium': '--',         'Low': '-.'}
ORDERED_4   = ['Severe', 'High', 'Medium', 'Low']
CMAP        = mcolors.LinearSegmentedColormap.from_list(
    "pub", ["#ffffff", "#d9d9d9", "#a6a6a6", "#737373", "#404040"])

Ready: Results_Honest
Ready: Results_Honest/Models
Ready: Results_Honest/Plots


## Step 1 — Load data

In [2]:
# =============================================================================
# STEP 1 — LOAD DATA
# =============================================================================
print("\n" + "="*70)
print("STEP 1 — LOADING DATA")
print("="*70)

df = pd.read_csv(DATA_PATH)
print(f"Loaded: {df.shape[0]} pipelines, {df.shape[1]} columns")
print(f"Confirmed failed pipelines (risk=1): {int(df['risk'].sum())}")
print(f"Class ratio (safe:failed): {int((df['risk']==0).sum())} : {int(df['risk'].sum())} "
      f"= {(df['risk']==0).sum()/df['risk'].sum():.1f}:1  "
      f"(predicting all-safe = {(df['risk']==0).sum()/len(df)*100:.1f}% accuracy — useless baseline)")
print(f"\nFailure count distribution:")
print(df['failure_count'].value_counts().sort_index().to_string())


STEP 1 — LOADING DATA
Loaded: 15331 pipelines, 14 columns
Confirmed failed pipelines (risk=1): 608
Class ratio (safe:failed): 14723 : 608 = 24.2:1  (predicting all-safe = 96.0% accuracy — useless baseline)

Failure count distribution:
failure_count
0    14723
1      546
2       47
3       11
4        2
5        1
7        1


## Step 1b — KDE feature distributions (Failed vs Safe)

In [3]:
# =============================================================================
# STEP 1b — KDE FEATURE DISTRIBUTION  (Failed vs Safe pipelines)
# 4-panel plot: one KDE curve per class per feature
# Blue filled curve  = safe pipelines   | Blue dashed line  = safe mean
# Red  filled curve  = failed pipelines | Red  dashed line  = failed mean
# Purple shading     = overlap zone (where classifier is uncertain)
# Mean value annotated next to each dashed line
# =============================================================================
print("\n" + "="*70)
print("STEP 1b — KDE FEATURE DISTRIBUTIONS  (Failed vs Safe)")
print("="*70)

from scipy.stats import gaussian_kde as _gkde

_failed = df[df['risk'] == 1.0]
_safe   = df[df['risk'] == 0.0]

_features = [
    ('max_operating_pressure', 'Max Operating Pressure (psi)', (0, 1400),
     'Most separable — failed lines run at higher pressure'),
    ('line_age_yr',            'Pipeline Age (years)',          (0, 55),
     'Failed lines are ~7 years older on average'),
    ('diameter_in',            'Diameter (inches)',             (0, 20),
     'Minimal separation — diameter alone is not predictive'),
    ('elevation',              'Elevation (m)',                 (900, 2800),
     'Failed lines at lower elevation (valley floor routes)'),
]

_CLR_F = COLOR_FAIL   # failed
_CLR_S = COLOR_SAFE   # safe

fig_kde, axes_kde = plt.subplots(2, 2, figsize=(18, 12))
axes_kde = axes_kde.flatten()

for ax, (col, label, xlim, insight) in zip(axes_kde, _features):
    f_vals = _failed[col].dropna()
    s_vals = _safe[col].dropna()

    x = np.linspace(xlim[0], xlim[1], 500)
    try:
        kde_f = _gkde(f_vals, bw_method='scott')
        kde_s = _gkde(s_vals, bw_method='scott')
        yf = kde_f(x)
        ys = kde_s(x)
    except Exception:
        ax.set_title('')
        continue

    y_peak = max(yf.max(), ys.max())

    # Filled KDE curves
    ax.fill_between(x, ys, alpha=0.20, color=_CLR_S)
    ax.fill_between(x, yf, alpha=0.22, color=_CLR_F)
    ax.plot(x, ys, color=_CLR_S, lw=1.8,
            label=f'Safe  (n={len(s_vals):,})')
    ax.plot(x, yf, color=_CLR_F, lw=2.2,
            label=f'Failed (n={len(f_vals):,})')

    # Purple overlap zone
    ax.fill_between(x, np.minimum(ys, yf), alpha=0.22, color=COLOR_OVERLAP,
                    label='Overlap zone')

    # ── Mean lines with value labels ──────────────────────────────────────
    s_mean = s_vals.mean()
    f_mean = f_vals.mean()

    ax.axvline(s_mean, color=_CLR_S, lw=1.8, ls='--', alpha=0.95,
               label=f'Safe mean = {s_mean:.1f}')
    ax.axvline(f_mean, color=_CLR_F, lw=1.8, ls='--', alpha=0.95,
               label=f'Failed mean = {f_mean:.1f}')

    # Annotate mean values directly on the plot
    ax.text(s_mean, y_peak * 0.72,
            f'Safe\nmean\n{s_mean:.1f}',
            color=_CLR_S, fontsize=FS_ANNOT,
            ha='right' if f_mean > s_mean else 'left',
            va='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=_CLR_S, alpha=0.85))

    ax.text(f_mean, y_peak * 0.72,
            f'Failed\nmean\n{f_mean:.1f}',
            color=_CLR_F, fontsize=FS_ANNOT,
            ha='left' if f_mean > s_mean else 'right',
            va='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                      edgecolor=_CLR_F, alpha=0.85))

    ax.set_xlim(xlim)
    ax.set_xlabel(label, fontsize=FS_LABEL)
    ax.set_ylabel('Density', fontsize=FS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    ax.legend(loc='best', fontsize=FS_LEGEND - 1,
              frameon=True, facecolor='white', edgecolor='#BDBDBD',
              framealpha=0.94)
    ax.set_title('')
    ax.grid(False)

    print(f"  {col:30s}  safe mean={s_mean:.1f}  failed mean={f_mean:.1f}")

add_bottom_note(
    fig_kde,
    'KDE Distribution: Failed vs Safe Pipelines | '
    f'{len(df):,} pipelines; {len(_failed):,} failed '
    f'({len(_failed)/len(df)*100:.1f}%) and {len(_safe):,} safe | '
    'Dashed lines show class means; purple shading marks overlap zones.'
)
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_kde_feature_distributions.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_kde_feature_distributions.png'))
plt.close()
print("Saved: fig_kde_feature_distributions")


STEP 1b — KDE FEATURE DISTRIBUTIONS  (Failed vs Safe)
  max_operating_pressure          safe mean=161.9  failed mean=328.7
  line_age_yr                     safe mean=12.0  failed mean=18.8
  diameter_in                     safe mean=4.4  failed mean=4.4
  elevation                       safe mean=1869.4  failed mean=1673.0


Saved: fig_kde_feature_distributions


## Step 2 — Binary ground-truth labels (Phase 1)

In [4]:
# =============================================================================
# STEP 2 — BINARY LABELS  (pure ground truth, no formula)
# =============================================================================
print("\n" + "="*70)
print("STEP 2 — BINARY LABELS  (Phase 1 — clean ground truth)")
print("="*70)

# Binary: High = confirmed failure, Low = no recorded failure
# Medium is NOT defined here — it emerges from the score distribution later
df['binary_label'] = (df['failure_count'] >= 1).astype(int)

n_high    = int(df['binary_label'].sum())
n_low     = len(df) - n_high
imb_ratio = n_low / n_high
print(f"  Failed (1) — confirmed failures : {n_high:6d}  ({n_high/len(df)*100:.1f}%)")
print(f"  Safe   (0) — no recorded failure: {n_low:6d}  ({n_low/len(df)*100:.1f}%)")
print(f"  Imbalance ratio: {imb_ratio:.1f}:1")

# Binary class distribution plot
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(['High (Failure)', 'Low (No Failure)'], [n_high, n_low],
              color=[COLOR_FAIL, COLOR_SAFE], edgecolor='black', linewidth=1.2, width=0.5)
for bar, cnt in zip(bars, [n_high, n_low]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{cnt}\n({cnt/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=FS_NUMBER)
ax.set_ylabel('Frequency', fontsize=FS_LABEL)
ax.set_xlabel('Binary Class (Training Labels)', fontsize=FS_LABEL)
ax.tick_params(labelsize=FS_TICK)
ax.set_ylim(0, max(n_high, n_low) * 1.2)
ax.grid(False)
add_bottom_note(fig, 'Binary class distribution used for training labels: High = confirmed failure; Low = no recorded failure.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_binary_class_distribution.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_binary_class_distribution.png'))
plt.close()
print("Saved: fig_binary_class_distribution")


STEP 2 — BINARY LABELS  (Phase 1 — clean ground truth)
  Failed (1) — confirmed failures :    608  (4.0%)
  Safe   (0) — no recorded failure:  14723  (96.0%)
  Imbalance ratio: 24.2:1


Saved: fig_binary_class_distribution


## Step 3 — Feature setup

All 11 features used, nothing excluded.

In [5]:
# =============================================================================
# STEP 3 — FEATURES  (all 11 features, nothing excluded)
# =============================================================================
print("\n" + "="*70)
print("STEP 3 — FEATURE SETUP  (all features included)")
print("="*70)

NUMERIC_FEATURES = [
    'line_age_yr',            # pipeline age in years
    'max_operating_pressure', # MAOP in psi
    'diameter_in',            # pipe diameter in inches
    'elevation',              # terrain elevation in metres
    'length_ft',              # physical pipeline length in feet
    'num_points',             # GIS 50m segment count — used for sample weighting only
                              # included in model input but excluded from importance chart
]
CATEGORICAL_FEATURES = [
    'status',           # Active / Abandoned / Out of Service
    'flowline_action',  # Regulatory action: Registration / Abandonment / Realignment
    'location_type',    # Connected facility type: Manifold / Well Site / Production Facilities
                        # NOT FLOWLINETYPE — that column was dropped during GIS processing
    'fluid',            # Crude Oil / Natural Gas / Multiphase / Produced Water
    'material',         # Carbon Steel / HDPE / Fiberglass / Steel
]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df['binary_label'].copy()

print(f"Features : {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical = {X.shape[1]} total")
print(f"Target   : binary_label  (1=failure, 0=no failure)")
print(f"Samples  : {len(X)}")


STEP 3 — FEATURE SETUP  (all features included)
Features : 6 numeric + 5 categorical = 11 total
Target   : binary_label  (1=failure, 0=no failure)
Samples  : 15331


## Step 4 — Train / Validation / Test split (60% / 20% / 20%) — **the fix**

A single 70/30 split let the test set be reused for model selection, threshold derivation, and
final reporting. This three-way split separates those concerns: only Train and Validation are
touched until Step 18.

In [6]:
# =============================================================================
# STEP 4 — TRAIN / VALIDATION / TEST SPLIT  60% / 20% / 20%
# A single 70/30 train/test split lets the test set be reused for model
# selection, threshold derivation, AND final reporting — three different
# decisions informed by the same held-out rows, which biases every metric
# derived from it optimistic. A three-way split fixes this: TRAIN fits the
# models; VALIDATION selects the best model, derives T2/T3, and decides
# whether calibration helps; TEST is touched exactly once, at the very end
# (Step 18), purely to report final honest generalisation numbers.
# =============================================================================
print("\n" + "="*70)
print("STEP 4 — TRAIN / VALIDATION / TEST SPLIT  (60% / 20% / 20%,  stratified)")
print("="*70)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=RAND_SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RAND_SEED, stratify=y_temp)

print(f"Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}")
print(f"Train failures: {y_train.sum()}  |  Val failures: {y_val.sum()}  |  Test failures: {y_test.sum()}")
print("NOTE: the test set above is not referenced again until Step 18.")


STEP 4 — TRAIN / VALIDATION / TEST SPLIT  (60% / 20% / 20%,  stratified)
Train: 9198  |  Val: 3066  |  Test: 3067
Train failures: 365  |  Val failures: 121  |  Test failures: 122
NOTE: the test set above is not referenced again until Step 18.


## Step 5 — Sample weights from `num_points`

In [7]:
# =============================================================================
# STEP 5 — SAMPLE WEIGHTS FROM num_points
# Each pipeline is represented by multiple 50m GIS segments.
# Longer pipelines (more segments) cover more infrastructure — they get
# higher weight during training. Weights are normalised so mean = 1.0.
# This is separate from class_weight balancing — it operates on top of it.
# num_points is NOT shown in charts because it has no independent physical
# significance; it is a GIS digitisation count, not a pipeline property.
# =============================================================================
print("\n" + "="*70)
print("STEP 5 — SAMPLE WEIGHTS FROM num_points")
print("="*70)

num_pts_train  = X_train['num_points'].fillna(X_train['num_points'].median())
sample_weights = num_pts_train / num_pts_train.mean()
print(f"Weight range : {sample_weights.min():.2f} – {sample_weights.max():.2f}")
print(f"Weight mean  : {sample_weights.mean():.2f}  (normalised to 1.0)")


STEP 5 — SAMPLE WEIGHTS FROM num_points
Weight range : 0.12 – 70.96
Weight mean  : 1.00  (normalised to 1.0)


## Step 6 — Preprocessing + two-stage imbalance correction

Preprocessor fit on **train only**; validation and test are both transformed with it (test not used again until Step 18).

In [8]:
# =============================================================================
# STEP 6 — PREPROCESSING + BorderlineSMOTE then TomekLinks (two-stage)
# NOTE: SMOTETomek only accepts basic SMOTE, not BorderlineSMOTE.
# So we run them as two separate steps which gives the same effect:
#   Step A — BorderlineSMOTE: oversample failures near the decision boundary
#   Step B — TomekLinks: remove overlapping majority samples near that boundary
# This cleans both sides of the boundary simultaneously.
# sampling_strategy=0.3 — failures brought to 30% of safe count.
# =============================================================================
print("\n" + "="*70)
print("STEP 6 — BorderlineSMOTE + TomekLinks  (two-stage boundary cleaning)")
print("="*70)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)  # not used again until Step 18

print(f"Before — Failed: {y_train.sum()}  Safe: {(y_train==0).sum()}")
bsmote = BorderlineSMOTE(sampling_strategy=0.3, random_state=RAND_SEED, k_neighbors=5)
X_res, y_res = bsmote.fit_resample(X_train_proc, y_train)
print(f"After BorderlineSMOTE — Failed: {y_res.sum()}  Safe: {(y_res==0).sum()}")

tomek = TomekLinks()
X_res, y_res = tomek.fit_resample(X_res, y_res)
print(f"After TomekLinks       — Failed: {y_res.sum()}  Safe: {(y_res==0).sum()}")
print(f"Total training samples: {len(X_res):,}")

# Sample weights — original samples keep their num_points weight,
# synthetic samples from SMOTE get weight 1.0 (mean weight).
# TomekLinks may remove some original samples — trim weights accordingly.
n_diff = len(X_res) - len(X_train_proc)
if n_diff >= 0:
    sw_res = np.concatenate([sample_weights.values, np.ones(n_diff)])
else:
    # TomekLinks removed more than SMOTE added — keep uniform weights
    sw_res = np.ones(len(X_res))


STEP 6 — BorderlineSMOTE + TomekLinks  (two-stage boundary cleaning)
Before — Failed: 365  Safe: 8833
After BorderlineSMOTE — Failed: 2649  Safe: 8833
After TomekLinks       — Failed: 2649  Safe: 8790
Total training samples: 11,439


## Step 6b — Leak-free stratified 5-fold cross-validation (train only, unaffected by the fix)

In [9]:
# =============================================================================
# STEP 6b — STRATIFIED 5-FOLD CV WITH SMOTE INSIDE FOLDS  (leak-free)
# Applying SMOTE before CV leaks information — synthetic samples from the same
# original pipeline appear in both train and validation folds, inflating metrics.
# Solution: use imblearn Pipeline so SMOTE runs only on the training fold.
# n_splits=5 gives ~85 failures per test fold — reliable estimates.
# =============================================================================
print("\n" + "="*70)
print("STEP 6b — STRATIFIED 5-FOLD CV  (SMOTE inside folds — no leakage)")
print("="*70)

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.calibration     import CalibratedClassifierCV, calibration_curve

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAND_SEED)

cv_models = {
    'Logistic Regression': ImbPipeline([
        ('smote', BorderlineSMOTE(sampling_strategy=0.3, random_state=RAND_SEED)),
        ('tomek', TomekLinks()),
        ('clf',   LogisticRegression(C=1, penalty='l2', solver='lbfgs',
                                     max_iter=1000, class_weight='balanced',
                                     random_state=RAND_SEED)),
    ]),
    'Random Forest': ImbPipeline([
        ('smote', BorderlineSMOTE(sampling_strategy=0.3, random_state=RAND_SEED)),
        ('tomek', TomekLinks()),
        ('clf',   RandomForestClassifier(n_estimators=150, max_depth=20,
                                         class_weight='balanced',
                                         random_state=RAND_SEED, n_jobs=-1)),
    ]),
    'XGBoost': ImbPipeline([
        ('smote', BorderlineSMOTE(sampling_strategy=0.3, random_state=RAND_SEED)),
        ('tomek', TomekLinks()),
        ('clf',   XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                subsample=0.8, colsample_bytree=0.8,
                                scale_pos_weight=imb_ratio, eval_metric='aucpr',
                                random_state=RAND_SEED, n_jobs=-1, verbosity=0)),
    ]),
}

cv_scoring = {
    'pr_auc':    'average_precision',
    'roc_auc':   'roc_auc',
    'f1_failed': 'f1',
    'bal_acc':   'balanced_accuracy',
    'recall':    'recall',
}

cv_results_summary = {}
print(f"\nRunning 5-fold CV (SMOTE inside each fold) ...")
for name, pipe in cv_models.items():
    print(f"  {name} ...", end=' ', flush=True)
    cv_res = cross_validate(pipe, X_train_proc, y_train,
                            cv=skf, scoring=cv_scoring,
                            n_jobs=-1, error_score='raise')
    cv_results_summary[name] = {
        k: (cv_res[f'test_{v}'].mean(), cv_res[f'test_{v}'].std())
        for k, v in [('pr_auc','pr_auc'), ('roc_auc','roc_auc'),
                     ('f1','f1_failed'), ('bal_acc','bal_acc'), ('recall','recall')]
    }
    r = cv_results_summary[name]
    print(f"PR-AUC={r['pr_auc'][0]:.4f}±{r['pr_auc'][1]:.4f}  "
          f"Recall={r['recall'][0]:.4f}±{r['recall'][1]:.4f}  "
          f"F1={r['f1'][0]:.4f}±{r['f1'][1]:.4f}")

print(f"\n{'─'*75}")
print(f"  {'Model':<22}  {'CV PR-AUC':>16}  {'CV Recall':>14}  "
      f"{'CV F1':>12}  {'CV Bal-Acc':>14}")
print(f"{'─'*75}")
for name, r in cv_results_summary.items():
    print(f"  {name:<22}  "
          f"{r['pr_auc'][0]:.4f}±{r['pr_auc'][1]:.3f}  "
          f"{r['recall'][0]:.4f}±{r['recall'][1]:.3f}  "
          f"{r['f1'][0]:.4f}±{r['f1'][1]:.3f}  "
          f"{r['bal_acc'][0]:.4f}±{r['bal_acc'][1]:.3f}")
print(f"{'─'*75}")
print("NOTE: CV metrics are leak-free — SMOTE applied inside each fold.")
print("      If hold-out metrics are much higher, the model is overfitting.\n")

cv_df = pd.DataFrame([{
    'Model':      name,
    'CV PR-AUC':  f"{r['pr_auc'][0]:.4f}±{r['pr_auc'][1]:.4f}",
    'CV Recall':  f"{r['recall'][0]:.4f}±{r['recall'][1]:.4f}",
    'CV F1':      f"{r['f1'][0]:.4f}±{r['f1'][1]:.4f}",
    'CV Bal-Acc': f"{r['bal_acc'][0]:.4f}±{r['bal_acc'][1]:.4f}",
    'CV ROC-AUC': f"{r['roc_auc'][0]:.4f}±{r['roc_auc'][1]:.4f}",
} for name, r in cv_results_summary.items()])
cv_df.to_csv(os.path.join(RESULTS_DIR, 'cv_performance_summary.csv'), index=False)
print(f"Saved: cv_performance_summary.csv")


STEP 6b — STRATIFIED 5-FOLD CV  (SMOTE inside folds — no leakage)

Running 5-fold CV (SMOTE inside each fold) ...
  Logistic Regression ... 

PR-AUC=0.2724±0.0726  Recall=0.8110±0.0293  F1=0.2375±0.0156
  Random Forest ... 

PR-AUC=0.5613±0.0206  Recall=0.5425±0.0186  F1=0.5295±0.0196
  XGBoost ... 

PR-AUC=0.6504±0.0060  Recall=0.6877±0.0329  F1=0.4645±0.0214

───────────────────────────────────────────────────────────────────────────
  Model                          CV PR-AUC       CV Recall         CV F1      CV Bal-Acc
───────────────────────────────────────────────────────────────────────────
  Logistic Regression     0.2724±0.073  0.8110±0.029  0.2375±0.016  0.8015±0.018
  Random Forest           0.5613±0.021  0.5425±0.019  0.5295±0.020  0.7608±0.010
  XGBoost                 0.6504±0.006  0.6877±0.033  0.4645±0.021  0.8175±0.017
───────────────────────────────────────────────────────────────────────────
NOTE: CV metrics are leak-free — SMOTE applied inside each fold.
      If hold-out metrics are much higher, the model is overfitting.

Saved: cv_performance_summary.csv


## Step 7 — Define the five candidate models

In [10]:
# =============================================================================
# STEP 7 — DEFINE MODELS  (all imbalance-aware)
# class_weight='balanced' on sklearn models — automatically sets weights
# inversely proportional to class frequency.
# XGBoost uses scale_pos_weight = n_safe/n_failed (~24.2) which tells
# XGBoost that each failed pipeline is 24x more important to classify correctly.
# =============================================================================
print("\n" + "="*70)
print("STEP 7 — DEFINING MODELS  (all imbalance-aware)")
print("="*70)

models = {
    'Logistic Regression': LogisticRegression(
        C=1, penalty='l2', solver='lbfgs',
        max_iter=1000, class_weight='balanced', random_state=RAND_SEED),

    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=20, min_samples_split=2,
        class_weight='balanced', random_state=RAND_SEED, n_jobs=-1),

    'SVM': SVC(
        C=10, gamma=0.1, kernel='rbf',
        probability=True, class_weight='balanced', random_state=RAND_SEED),

    'XGBoost': XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=imb_ratio,   # penalises missing failures ~24x more
        eval_metric='aucpr',          # optimises on PR-AUC during training
        random_state=RAND_SEED, n_jobs=-1, verbosity=0),

    'Stacking Ensemble': StackingClassifier(
        estimators=[
            ('gb',  GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                               max_depth=3, random_state=RAND_SEED)),
            ('rf',  RandomForestClassifier(n_estimators=100, max_depth=10,
                                           class_weight='balanced',
                                           random_state=RAND_SEED, n_jobs=-1)),
            ('svm', SVC(C=10, gamma=0.1, kernel='rbf', probability=True,
                        class_weight='balanced', random_state=RAND_SEED)),
        ],
        final_estimator=LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=RAND_SEED),
        cv=5, n_jobs=-1),
}
print("Models:", list(models.keys()))
print(f"XGBoost scale_pos_weight = {imb_ratio:.1f}  (n_safe / n_failed)")


STEP 7 — DEFINING MODELS  (all imbalance-aware)
Models: ['Logistic Regression', 'Random Forest', 'SVM', 'XGBoost', 'Stacking Ensemble']
XGBoost scale_pos_weight = 24.2  (n_safe / n_failed)


## Step 8 — Train and evaluate — **now on VALIDATION, not test**

Models are fit on the resampled training data and evaluated on the validation split. This is used purely for model comparison/selection; the test set remains untouched.

In [11]:
# =============================================================================
# STEP 8 — TRAIN AND EVALUATE
# Primary metric: PR-AUC (average_precision_score)
# Accuracy alone is misleading — predicting all-safe gives 96% accuracy
# =============================================================================
print("\n" + "="*70)
print("STEP 8 — TRAINING AND EVALUATION  (on VALIDATION set — test set untouched)")
print("  Primary metric: PR-AUC  |  Accuracy alone is misleading for 96:4 data")
print("="*70)

MODEL_COLORS = {
    'Logistic Regression': COLOR_SAFE,
    'Random Forest':       COLOR_FAIL,
    'SVM':                 COLOR_MEDIUM,
    'XGBoost':             COLOR_SUCCESS,
    'Stacking Ensemble':   COLOR_OVERLAP,
}
MODEL_LS = {
    'Logistic Regression': '--',
    'Random Forest':       '-',
    'SVM':                 '-.',
    'XGBoost':             '-',
    'Stacking Ensemble':   ':',
}

results = {}
for name, model in models.items():
    print(f"\n{'─'*60}\nTraining: {name}\n{'─'*60}")

    # XGBoost receives sample_weight directly — other models use resampled data
    if name == 'XGBoost':
        model.fit(X_res, y_res, sample_weight=sw_res)
    else:
        model.fit(X_res, y_res)

    y_pred  = model.predict(X_val_proc)
    y_prob  = model.predict_proba(X_val_proc)[:, 1]

    acc     = accuracy_score(y_val, y_pred)
    bal_acc = balanced_accuracy_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_prob)
    pr_auc  = average_precision_score(y_val, y_prob)
    f1_fail = f1_score(y_val, y_pred, pos_label=1)
    report  = classification_report(y_val, y_pred,
                                    target_names=['Safe', 'Failed'],
                                    output_dict=True)

    results[name] = {
        'model':       model,
        'y_pred':      y_pred,
        'y_prob':      y_prob,
        'accuracy':    acc,
        'bal_acc':     bal_acc,
        'roc_auc':     roc_auc,
        'pr_auc':      pr_auc,
        'f1_fail':     f1_fail,
        'recall_fail': report['Failed']['recall'],
        'report':      report,
    }
    print(f"  Accuracy (misleading) : {acc:.4f}")
    print(f"  Balanced Accuracy     : {bal_acc:.4f}")
    print(f"  ROC-AUC               : {roc_auc:.4f}")
    print(f"  PR-AUC (PRIMARY)      : {pr_auc:.4f}")
    print(f"  F1 — Failed class     : {f1_fail:.4f}")
    print(classification_report(y_val, y_pred, target_names=['Safe', 'Failed']))


STEP 8 — TRAINING AND EVALUATION  (on VALIDATION set — test set untouched)
  Primary metric: PR-AUC  |  Accuracy alone is misleading for 96:4 data

────────────────────────────────────────────────────────────
Training: Logistic Regression
────────────────────────────────────────────────────────────
  Accuracy (misleading) : 0.7818
  Balanced Accuracy     : 0.7794
  ROC-AUC               : 0.8665
  PR-AUC (PRIMARY)      : 0.2292
  F1 — Failed class     : 0.2194
              precision    recall  f1-score   support

        Safe       0.99      0.78      0.87      2945
      Failed       0.13      0.78      0.22       121

    accuracy                           0.78      3066
   macro avg       0.56      0.78      0.55      3066
weighted avg       0.95      0.78      0.85      3066


────────────────────────────────────────────────────────────
Training: Random Forest
────────────────────────────────────────────────────────────


  Accuracy (misleading) : 0.9586
  Balanced Accuracy     : 0.7724
  ROC-AUC               : 0.9164
  PR-AUC (PRIMARY)      : 0.5767
  F1 — Failed class     : 0.5208
              precision    recall  f1-score   support

        Safe       0.98      0.97      0.98      2945
      Failed       0.48      0.57      0.52       121

    accuracy                           0.96      3066
   macro avg       0.73      0.77      0.75      3066
weighted avg       0.96      0.96      0.96      3066


────────────────────────────────────────────────────────────
Training: SVM
────────────────────────────────────────────────────────────


  Accuracy (misleading) : 0.9145
  Balanced Accuracy     : 0.8049
  ROC-AUC               : 0.8893
  PR-AUC (PRIMARY)      : 0.4147
  F1 — Failed class     : 0.3879
              precision    recall  f1-score   support

        Safe       0.99      0.92      0.95      2945
      Failed       0.27      0.69      0.39       121

    accuracy                           0.91      3066
   macro avg       0.63      0.80      0.67      3066
weighted avg       0.96      0.91      0.93      3066


────────────────────────────────────────────────────────────
Training: XGBoost
────────────────────────────────────────────────────────────


  Accuracy (misleading) : 0.9295
  Balanced Accuracy     : 0.8484
  ROC-AUC               : 0.9482
  PR-AUC (PRIMARY)      : 0.6894
  F1 — Failed class     : 0.4600
              precision    recall  f1-score   support

        Safe       0.99      0.94      0.96      2945
      Failed       0.33      0.76      0.46       121

    accuracy                           0.93      3066
   macro avg       0.66      0.85      0.71      3066
weighted avg       0.96      0.93      0.94      3066


────────────────────────────────────────────────────────────
Training: Stacking Ensemble
────────────────────────────────────────────────────────────


  Accuracy (misleading) : 0.9446
  Balanced Accuracy     : 0.8047
  ROC-AUC               : 0.9268
  PR-AUC (PRIMARY)      : 0.5804
  F1 — Failed class     : 0.4817
              precision    recall  f1-score   support

        Safe       0.99      0.96      0.97      2945
      Failed       0.38      0.65      0.48       121

    accuracy                           0.94      3066
   macro avg       0.68      0.80      0.73      3066
weighted avg       0.96      0.94      0.95      3066



## Step 9 — Validation performance summary table (ranked by PR-AUC)

In [12]:
# =============================================================================
# STEP 9 — PERFORMANCE SUMMARY TABLE  (sorted by PR-AUC)
# =============================================================================
print("\n" + "="*70)
print("STEP 9 — VALIDATION PERFORMANCE SUMMARY TABLE  (sorted by PR-AUC)")
print("="*70)

rows = []
for name, res in results.items():
    rows.append({
        'Model':          name,
        'PR-AUC':         round(res['pr_auc'],     4),
        'ROC-AUC':        round(res['roc_auc'],    4),
        'Balanced Acc':   round(res['bal_acc'],     4),
        'Failed Recall':  round(res['recall_fail'], 4),
        'Failed F1':      round(res['f1_fail'],     4),
        'Accuracy':       round(res['accuracy'],    4),
    })
summary_df = pd.DataFrame(rows).sort_values('PR-AUC', ascending=False)
print(summary_df.to_string(index=False))
out_csv = os.path.join(RESULTS_DIR, 'model_performance_summary_validation.csv')
summary_df.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")
print("NOTE: Best model selected by PR-AUC — not accuracy.")


STEP 9 — VALIDATION PERFORMANCE SUMMARY TABLE  (sorted by PR-AUC)
              Model  PR-AUC  ROC-AUC  Balanced Acc  Failed Recall  Failed F1  Accuracy
            XGBoost  0.6894   0.9482        0.8484         0.7603     0.4600    0.9295
  Stacking Ensemble  0.5804   0.9268        0.8047         0.6529     0.4817    0.9446
      Random Forest  0.5767   0.9164        0.7724         0.5702     0.5208    0.9586
                SVM  0.4147   0.8893        0.8049         0.6860     0.3879    0.9145
Logistic Regression  0.2292   0.8665        0.7794         0.7769     0.2194    0.7818

Saved: Results_Honest/model_performance_summary_validation.csv
NOTE: Best model selected by PR-AUC — not accuracy.


## Step 10 — Optimal decision threshold per model — **derived on VALIDATION**

This is the threshold later reused as `T3`. Previously derived from the test set (test-set leakage); now derived purely from validation.

In [13]:
# =============================================================================
# STEP 10 — OPTIMAL THRESHOLD FROM PRECISION-RECALL CURVE
# Default threshold=0.5 is wrong for 96:4 imbalanced data.
# Best threshold = point on PR curve that maximises F1 for the failed class.
# =============================================================================
print("\n" + "="*70)
print("STEP 10 — OPTIMAL THRESHOLD FROM PR CURVE  (derived on VALIDATION set)")
print("="*70)

for name, res in results.items():
    prec, rec, thresholds = precision_recall_curve(y_val, res['y_prob'])
    f1_arr   = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-8)
    best_idx = f1_arr.argmax()
    best_t   = thresholds[best_idx]

    y_pred_opt = (res['y_prob'] >= best_t).astype(int)
    results[name]['best_threshold'] = best_t
    results[name]['prec_at_best']   = prec[best_idx]
    results[name]['rec_at_best']    = rec[best_idx]
    results[name]['y_pred_opt']     = y_pred_opt
    results[name]['f1_fail_opt']    = f1_score(y_val, y_pred_opt, pos_label=1)
    results[name]['recall_opt']     = rec[best_idx]

    print(f"\n  {name}:")
    print(f"    Default  thr=0.50  →  F1={res['f1_fail']:.4f}  "
          f"Recall={res['recall_fail']:.4f}")
    print(f"    Optimal  thr={best_t:.4f}  →  F1={f1_arr[best_idx]:.4f}  "
          f"Prec={prec[best_idx]:.4f}  Rec={rec[best_idx]:.4f}")


STEP 10 — OPTIMAL THRESHOLD FROM PR CURVE  (derived on VALIDATION set)

  Logistic Regression:
    Default  thr=0.50  →  F1=0.2194  Recall=0.7769
    Optimal  thr=0.7801  →  F1=0.3309  Prec=0.2359  Rec=0.5537

  Random Forest:
    Default  thr=0.50  →  F1=0.5208  Recall=0.5702
    Optimal  thr=0.5599  →  F1=0.5593  Prec=0.5739  Rec=0.5455

  SVM:
    Default  thr=0.50  →  F1=0.3879  Recall=0.6860
    Optimal  thr=0.7204  →  F1=0.4883  Prec=0.5652  Rec=0.4298

  XGBoost:
    Default  thr=0.50  →  F1=0.4600  Recall=0.7603
    Optimal  thr=0.8723  →  F1=0.6407  Prec=0.6727  Rec=0.6116

  Stacking Ensemble:
    Default  thr=0.50  →  F1=0.4817  Recall=0.6529
    Optimal  thr=0.8222  →  F1=0.5690  Prec=0.5946  Rec=0.5455


## Step 11 — Confusion matrices (optimal threshold, VALIDATION set)

In [14]:
# =============================================================================
# STEP 11 — CONFUSION MATRICES  (optimal threshold, Safe/Failed labels)
# =============================================================================
print("\n" + "="*70)
print("STEP 11 — CONFUSION MATRICES  (optimal threshold, VALIDATION set)")
print("="*70)

for name, res in results.items():
    cm_mat = confusion_matrix(y_val, res['y_pred_opt'])
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(confusion_matrix=cm_mat,
                           display_labels=['Safe', 'Failed']).plot(
        ax=ax, cmap=CMAP, colorbar=False, values_format='d',
        text_kw={"fontsize": FS_NUMBER})
    ax.set_title('')
    ax.set_xlabel('Predicted', fontsize=FS_LABEL)
    ax.set_ylabel('Actual',    fontsize=FS_LABEL)
    ax.tick_params(labelsize=FS_TICK)
    add_bottom_note(fig, f'{name} confusion matrix | Threshold={res["best_threshold"]:.2f}; PR-AUC={res["pr_auc"]:.3f}')
    plt.tight_layout(rect=[0, 0.10, 1, 1])
    fname = name.lower().replace(' ', '_')
    plt.savefig(os.path.join(PLOTS_DIR, f'confusion_matrix_{fname}.pdf'))
    plt.savefig(os.path.join(PLOTS_DIR, f'confusion_matrix_{fname}.png'))
    plt.close()
    print(f"Saved: confusion_matrix_{fname}")


STEP 11 — CONFUSION MATRICES  (optimal threshold, VALIDATION set)


Saved: confusion_matrix_logistic_regression


Saved: confusion_matrix_random_forest


Saved: confusion_matrix_svm


Saved: confusion_matrix_xgboost


Saved: confusion_matrix_stacking_ensemble


## Step 12 — Precision-Recall and ROC curves (VALIDATION set)

In [15]:
# =============================================================================
# STEP 12 — PRECISION-RECALL + ROC CURVES
# PR-AUC is the primary evaluation plot for imbalanced data.
# ROC is included as secondary context.
# =============================================================================
print("\n" + "="*70)
print("STEP 12 — PRECISION-RECALL + ROC CURVES  (VALIDATION set)")
print("="*70)

fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(18, 8.5))

# PR curves — primary
baseline_pr = n_high / len(y_val)
ax_pr.axhline(baseline_pr, color=COLOR_NEUTRAL, lw=1.5, ls='--',
              label=f'No-skill baseline ({baseline_pr:.3f})')
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_val, res['y_prob'])
    ax_pr.plot(rec, prec,
               color=MODEL_COLORS[name], linestyle=MODEL_LS[name],
               lw=2.2, label=f'{name}  (PR-AUC={res["pr_auc"]:.3f})')
    ax_pr.scatter(res['rec_at_best'], res['prec_at_best'],
                  color=MODEL_COLORS[name], s=80, zorder=5, marker='D')
ax_pr.set_xlabel('Recall  (Failed class)', fontsize=FS_LABEL)
ax_pr.set_ylabel('Precision  (Failed class)', fontsize=FS_LABEL)
ax_pr.set_title('')
ax_pr.tick_params(labelsize=FS_TICK)
ax_pr.legend(loc='upper right',
             fontsize=FS_LEGEND, ncol=1,
             frameon=True, facecolor='white', edgecolor='#BDBDBD',
             framealpha=0.94)
ax_pr.grid(False)
ax_pr.set_xlim(0, 1); ax_pr.set_ylim(0, 1.05)

# ROC curves — secondary
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_val, res['y_prob'])
    ax_roc.plot(fpr, tpr,
                color=MODEL_COLORS[name], linestyle=MODEL_LS[name],
                lw=2.2, label=f'{name}  (AUC={res["roc_auc"]:.3f})')
ax_roc.plot([0,1],[0,1], 'r--', lw=1.5, label='Random chance')
ax_roc.set_xlabel('False Positive Rate', fontsize=FS_LABEL)
ax_roc.set_ylabel('True Positive Rate', fontsize=FS_LABEL)
ax_roc.set_title('')
ax_roc.tick_params(labelsize=FS_TICK)
ax_roc.legend(loc='lower right',
              fontsize=FS_LEGEND, ncol=1,
              frameon=True, facecolor='white', edgecolor='#BDBDBD',
              framealpha=0.94)
ax_roc.grid(False)

add_bottom_note(fig, 'Precision-Recall and ROC curves | PR-AUC is the primary metric for imbalanced data; diamonds mark optimal threshold points.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_pr_roc_curves.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_pr_roc_curves.png'))
plt.close()
print("Saved: fig_pr_roc_curves")


STEP 12 — PRECISION-RECALL + ROC CURVES  (VALIDATION set)


Saved: fig_pr_roc_curves


## Step 13 — Feature importance (Random Forest)

In [16]:
# =============================================================================
# STEP 13 — FEATURE IMPORTANCE  (Random Forest)
# num_points EXCLUDED from chart — GIS segment count, no physical independence.
# Corrected labels:
#   location_type   → 'Connected Facility Type'  (not 'Flowline Type')
#   flowline_action → 'Regulatory Action'
# =============================================================================
print("\n" + "="*70)
print("STEP 13 — FEATURE IMPORTANCE  (Random Forest, class_weight=balanced)")
print("  num_points excluded — GIS count, not an independent physical feature")
print("="*70)

rf_model  = results['Random Forest']['model']
ohe_feats = list(preprocessor
                 .named_transformers_['cat']
                 .named_steps['encoder']
                 .get_feature_names_out(CATEGORICAL_FEATURES))
all_feats = NUMERIC_FEATURES + ohe_feats
imps      = rf_model.feature_importances_

imp_dict = {}
for feat in NUMERIC_FEATURES:
    imp_dict[feat] = imps[all_feats.index(feat)]
for cat in CATEGORICAL_FEATURES:
    idxs = [i for i, f in enumerate(all_feats) if f.startswith(cat)]
    imp_dict[cat] = sum(imps[i] for i in idxs)

label_map = {
    'line_age_yr':            'Age (years)',
    'max_operating_pressure': 'MAOP (psi)',
    'diameter_in':            'Diameter (inches)',
    'elevation':              'Elevation (m)',
    'length_ft':              'Length (ft)',
    'status':                 'Status',
    'flowline_action':        'Regulatory Action',
    'location_type':          'Connected Facility Type',
    'fluid':                  'Fluid Type',
    'material':               'Pipe Material',
    # num_points intentionally excluded
}
imp_df = pd.DataFrame([
    {'Feature': label_map.get(k, k), 'Importance': v}
    for k, v in imp_dict.items()
    if k != 'num_points'          # exclude from chart
]).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.barh(imp_df['Feature'], imp_df['Importance'],
               color=COLOR_SAFE, edgecolor='black', linewidth=0.8)
max_imp = imp_df['Importance'].max()
for bar, val in zip(bars, imp_df['Importance']):
    ax.text(val + max_imp * 0.025, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=FS_NUMBER)
ax.set_xlabel('Feature Importance (Random Forest — Binary)', fontsize=FS_LABEL)
ax.tick_params(labelsize=FS_TICK)
ax.set_xlim(0, max_imp * 1.22)
ax.grid(False)
add_bottom_note(fig, 'Feature importance from the Random Forest binary model; num_points is excluded because it is a GIS segment count.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_feature_importance.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_feature_importance.png'))
plt.close()
print("Saved: fig_feature_importance")


STEP 13 — FEATURE IMPORTANCE  (Random Forest, class_weight=balanced)
  num_points excluded — GIS count, not an independent physical feature


Saved: fig_feature_importance


## Step 14 — Continuous risk score (Phase 2) — calibration decision now on VALIDATION

The best model is selected by **validation** PR-AUC. The decision of whether isotonic
calibration helps is also made on validation, not test — using both PR-AUC and the Brier
score (a direct reliability diagnostic) as evidence.

In [17]:
# =============================================================================
# STEP 14 — CONTINUOUS RISK SCORE  (Phase 2)
# Best model selected by PR-AUC — not accuracy or ROC-AUC
# =============================================================================
print("\n" + "="*70)
print("STEP 14 — CONTINUOUS RISK SCORE  (Phase 2)")
print("  Best model selected by PR-AUC — not ROC-AUC or accuracy")
print("="*70)

best_name  = max(results, key=lambda x: results[x]['pr_auc'])
best_model = results[best_name]['model']
print(f"Best model (by PR-AUC): {best_name}")
print(f"  PR-AUC           : {results[best_name]['pr_auc']:.4f}")
print(f"  ROC-AUC          : {results[best_name]['roc_auc']:.4f}")
print(f"  Balanced Acc     : {results[best_name]['bal_acc']:.4f}")
print(f"  Failed Recall    : {results[best_name]['recall_opt']:.4f}")
print(f"  Optimal threshold: {results[best_name]['best_threshold']:.4f}")

# =============================================================================
# PROBABILITY CALIBRATION
# Raw model probabilities may not reflect true failure likelihood.
# CalibratedClassifierCV with isotonic regression corrects the mapping
# between predicted scores and actual failure rates.
# If calibration does not hurt PR-AUC by more than 0.01, the calibrated
# model is used for all risk score generation.
# =============================================================================
print(f"\nCalibrating {best_name} probabilities (isotonic regression, cv=5, fit on TRAIN) ...")
calibrated_model = CalibratedClassifierCV(best_model, method='isotonic', cv=5)
calibrated_model.fit(X_train_proc, y_train)

y_prob_uncal = best_model.predict_proba(X_val_proc)[:, 1]
y_prob_cal   = calibrated_model.predict_proba(X_val_proc)[:, 1]
pr_auc_uncal = average_precision_score(y_val, y_prob_uncal)
pr_auc_cal   = average_precision_score(y_val, y_prob_cal)
print(f"  PR-AUC before calibration (VALIDATION): {pr_auc_uncal:.4f}")
print(f"  PR-AUC after  calibration (VALIDATION): {pr_auc_cal:.4f}")

if pr_auc_cal >= pr_auc_uncal - 0.01:
    scoring_model = calibrated_model
    results[best_name]['calibrated'] = True
    print(f"  Using CALIBRATED model for risk scores")
else:
    scoring_model = best_model
    results[best_name]['calibrated'] = False
    print(f"  Calibration hurt PR-AUC — keeping uncalibrated model")

# =============================================================================
# BRIER SCORE — direct reliability diagnostic, requested alongside/instead of
# the calibration curve. Lower is better; Brier score decomposes into
# calibration error + refinement, so unlike PR-AUC it directly penalises
# miscalibrated probabilities rather than just ranking quality.
# =============================================================================
brier_uncal = brier_score_loss(y_val, y_prob_uncal)
brier_cal   = brier_score_loss(y_val, y_prob_cal)
print(f"  Brier score before calibration (VALIDATION): {brier_uncal:.4f}")
print(f"  Brier score after  calibration (VALIDATION): {brier_cal:.4f}")
print(f"  Relative reduction: {(1 - brier_cal/brier_uncal)*100:.1f}%")

# Calibration curve plot
fig, ax = plt.subplots(figsize=(8, 6.5))
prob_true_u, prob_pred_u = calibration_curve(y_val, y_prob_uncal, n_bins=10)
prob_true_c, prob_pred_c = calibration_curve(y_val, y_prob_cal,   n_bins=10)
ax.plot([0,1],[0,1], 'k--', lw=1.5, label='Perfect calibration')
ax.plot(prob_pred_u, prob_true_u, 's-', color=COLOR_FAIL, lw=2,
        label=f'Before  (Brier={brier_uncal:.3f}, PR-AUC={pr_auc_uncal:.3f})')
ax.plot(prob_pred_c, prob_true_c, 'o-', color=COLOR_SUCCESS, lw=2,
        label=f'After   (Brier={brier_cal:.3f}, PR-AUC={pr_auc_cal:.3f})')
ax.set_xlabel('Mean Predicted Probability', fontsize=FS_LABEL)
ax.set_ylabel('Fraction of Positives  (actual failure rate)', fontsize=FS_LABEL)
ax.set_title('')
ax.tick_params(labelsize=FS_TICK)
ax.legend(loc='lower right', fontsize=FS_LEGEND,
          frameon=True, facecolor='white', edgecolor='#BDBDBD',
          framealpha=0.94)
ax.grid(False)
add_bottom_note(fig, f'Calibration curve for {best_name} (validation set) | Brier score is the direct reliability diagnostic — lower is better; closer to the diagonal means more trustworthy risk scores.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_calibration_curve.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_calibration_curve.png'))
plt.close()
print("Saved: fig_calibration_curve")

# =============================================================================
# OPERATIONAL REGISTRY SCORING — NOT a generalisation metric.
# The lines below score the ENTIRE 15,331-pipeline registry (train+val+test
# combined) with the final locked model, exactly as a deployed risk-scoring
# tool would: every real pipeline gets a score, including ones the model was
# trained on. df['risk_score'] and the Low/Medium/High/Severe labels derived
# from it below (Steps 13-16) describe the operational registry — they are
# NOT a measure of how well the model generalises to unseen pipelines.
# For the honest, leak-free generalisation numbers (thresholds derived purely
# from Train/Val, scored on TEST exactly once), see Step 18.
# =============================================================================
X_all_proc           = preprocessor.transform(X)
df['risk_score']     = scoring_model.predict_proba(X_all_proc)[:, 1]

# Track which split each pipeline belonged to, for transparency/reproducibility.
df['split'] = 'train'
df.loc[X_val.index, 'split'] = 'val'
df.loc[X_test.index, 'split'] = 'test'

# Store each model's individual scores
for name, res in results.items():
    col = 'score_' + name.lower().replace(' ', '_')
    df[col] = res['model'].predict_proba(X_all_proc)[:, 1]

print(f"\nRisk score statistics (all {len(df)} pipelines, OPERATIONAL registry scoring):")
print(df['risk_score'].describe().round(4))

print(f"\nRisk score by true binary label (OPERATIONAL — includes trained-on rows):")
print(df.groupby('binary_label')['risk_score'].describe().round(4))


STEP 14 — CONTINUOUS RISK SCORE  (Phase 2)
  Best model selected by PR-AUC — not ROC-AUC or accuracy
Best model (by PR-AUC): XGBoost
  PR-AUC           : 0.6894
  ROC-AUC          : 0.9482
  Balanced Acc     : 0.8484
  Failed Recall    : 0.6116
  Optimal threshold: 0.8723

Calibrating XGBoost probabilities (isotonic regression, cv=5, fit on TRAIN) ...


  PR-AUC before calibration (VALIDATION): 0.6894
  PR-AUC after  calibration (VALIDATION): 0.6800
  Using CALIBRATED model for risk scores
  Brier score before calibration (VALIDATION): 0.0490
  Brier score after  calibration (VALIDATION): 0.0203
  Relative reduction: 58.5%


Saved: fig_calibration_curve



Risk score statistics (all 15331 pipelines, OPERATIONAL registry scoring):
count    15331.0000
mean         0.0447
std          0.1452
min          0.0000
25%          0.0000
50%          0.0039
75%          0.0182
max          1.0000
Name: risk_score, dtype: float64

Risk score by true binary label (OPERATIONAL — includes trained-on rows):
                count    mean     std     min     25%     50%     75%     max
binary_label                                                                 
0             14723.0  0.0202  0.0492  0.0000  0.0000  0.0036  0.0129  0.8596
1               608.0  0.6392  0.3245  0.0028  0.4009  0.7184  0.9667  1.0000


## Step 15 — Score distribution and threshold derivation (Phase 3)

`df['risk_score']` scores the **entire 15,331-pipeline registry** (train+val+test combined)
with the final locked model — this is the operational registry scoring, explicitly labeled as
such in the code. It is not a generalisation metric; see Step 18 for that.

In [18]:
# =============================================================================
# STEP 13 — SCORE DISTRIBUTION PLOT + THRESHOLD DERIVATION  (Phase 3)
# Fit Gaussian KDE to score distribution
# T1 = mean - 1*std  (boundary between Low and Medium)
# T2 = mean + 1*std  (boundary between Medium and High)
# =============================================================================
print("\n" + "="*70)
print("STEP 13 — SCORE DISTRIBUTION + THRESHOLD DERIVATION  (Phase 3)")
print("="*70)

scores     = df['risk_score'].values
score_mean = scores.mean()
score_std  = scores.std()

# Auto thresholds (can be overridden at top of script)
T1 = T1_OVERRIDE if T1_OVERRIDE is not None else max(0.01, score_mean - score_std)
T2 = T2_OVERRIDE if T2_OVERRIDE is not None else 0.30
# T3 = High|Severe boundary — PR-optimal threshold from test set
# This is where the model is most confident about failure
T3 = T3_OVERRIDE if T3_OVERRIDE is not None else results[best_name]['best_threshold']

print(f"Score distribution — mean: {score_mean:.4f}  std: {score_std:.4f}")
print(f"T1 (Low    | Medium boundary) : {T1:.4f}  — below safe population")
print(f"T2 (Medium | High   boundary) : {T2:.4f}  — F1-optimal, ≥30% failure probability")
print(f"T3 (High   | Severe boundary) : {T3:.4f}  — PR-optimal, model is most confident")

# Separate scores by true label
scores_safe = df.loc[df['binary_label']==0, 'risk_score'].values
scores_fail = df.loc[df['binary_label']==1, 'risk_score'].values

x_grid = np.linspace(0, 1, 500)

# KDE for each population
kde_all  = stats.gaussian_kde(scores,      bw_method=0.15)
kde_safe = stats.gaussian_kde(scores_safe, bw_method=0.15)
kde_fail = stats.gaussian_kde(scores_fail, bw_method=0.10)

kde_all_v  = kde_all(x_grid)
kde_safe_v = kde_safe(x_grid)
kde_fail_v = kde_fail(x_grid)


STEP 13 — SCORE DISTRIBUTION + THRESHOLD DERIVATION  (Phase 3)
Score distribution — mean: 0.0447  std: 0.1452
T1 (Low    | Medium boundary) : 0.0500  — below safe population
T2 (Medium | High   boundary) : 0.3000  — F1-optimal, ≥30% failure probability
T3 (High   | Severe boundary) : 0.8723  — PR-optimal, model is most confident


### Plot A — Combined score distribution with the four threshold zones (operational registry)

In [19]:
# ── PLOT A: Combined distribution with threshold zones (original, improved) ──
fig, ax = plt.subplots(figsize=(14, 7))

ax.fill_between(x_grid, kde_all_v,
                where=(x_grid < T1),
                color=COLOR_SAFE, alpha=0.22, label=f'Low  (score < {T1:.2f})')
ax.fill_between(x_grid, kde_all_v,
                where=((x_grid >= T1) & (x_grid < T2)),
                color=COLOR_MEDIUM, alpha=0.22,
                label=f'Medium  ({T1:.2f} ≤ score < {T2:.2f})')
ax.fill_between(x_grid, kde_all_v,
                where=((x_grid >= T2) & (x_grid < T3)),
                color=COLOR_FAIL, alpha=0.22, label=f'High  ({T2:.2f} ≤ score < {T3:.2f})')
ax.fill_between(x_grid, kde_all_v,
                where=(x_grid >= T3),
                color=COLOR_SEVERE, alpha=0.28, label=f'Severe  (score ≥ {T3:.2f})')
ax.plot(x_grid, kde_all_v, color='black', linewidth=1.8, label='All pipelines (KDE)')

ax.axvline(x=T1, color=COLOR_SAFE, linewidth=1.8, linestyle='--')
ax.axvline(x=T2, color=COLOR_FAIL, linewidth=1.8, linestyle='--')
ax.axvline(x=T3, color=COLOR_SEVERE, linewidth=1.8, linestyle='-.')

ymax = kde_all_v.max()
t3_label_x, t3_label_ha = threshold_label_position(T3)
ax.text(T1 + 0.01, ymax * 0.88, f'T1={T1:.2f}',
        color=TEXT_SAFE, fontsize=FS_ANNOT, bbox=ANNOT_BOX)
ax.text(T2 + 0.01, ymax * 0.78, f'T2={T2:.2f}',
        color=TEXT_FAIL, fontsize=FS_ANNOT, bbox=ANNOT_BOX)
ax.text(t3_label_x, ymax * 0.52, f'T3={T3:.2f}',
        color=TEXT_SEVERE, fontsize=FS_ANNOT, ha=t3_label_ha,
        bbox=ANNOT_BOX)

# Rug plot for confirmed failures
ax.plot(scores_fail, np.full_like(scores_fail, -ymax * 0.02),
        '|', color=COLOR_FAIL, alpha=0.5, markersize=10,
        label=f'Confirmed failures  (n={len(scores_fail)})')

ax.set_xlabel('Risk Score  (predicted probability of failure)', fontsize=FS_LABEL)
ax.set_ylabel('Density', fontsize=FS_LABEL)
ax.tick_params(labelsize=FS_TICK)
ax.set_xlim(0, 1)
ax.set_ylim(bottom=-ymax * 0.05)
ax.legend(loc='upper right',
          fontsize=FS_LEGEND,
          frameon=True, facecolor='white', edgecolor='#BDBDBD',
          framealpha=0.94)
ax.grid(False)
add_bottom_note(fig, 'Risk score distribution for all pipelines | Shaded zones mark Low, Medium, High, and Severe score bands.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_risk_score_distribution.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_risk_score_distribution.png'))
plt.close()
print("Saved: fig_risk_score_distribution")

Saved: fig_risk_score_distribution


### Plot B — Safe vs. confirmed-failure populations, same score axis (operational registry)

In [20]:
# ── PLOT B: Split KDE — safe vs failure populations on same axes ──
# This is the key visualisation — shows the two populations are separated
fig, ax = plt.subplots(figsize=(14, 7))

ax.fill_between(x_grid, kde_safe_v, color=COLOR_SAFE, alpha=0.20,
                label=f'Safe pipelines  (n={len(scores_safe):,})')
ax.fill_between(x_grid, kde_fail_v, color=COLOR_FAIL, alpha=0.22,
                label=f'Confirmed failures  (n={len(scores_fail)})')

ax.plot(x_grid, kde_safe_v, color=COLOR_SAFE, linewidth=2.2, linestyle='-')
ax.plot(x_grid, kde_fail_v, color=COLOR_FAIL, linewidth=2.2, linestyle='-')

ax.axvline(x=T1, color=COLOR_SAFE, linewidth=1.8, linestyle='--', alpha=0.8)
ax.axvline(x=T2, color=COLOR_FAIL, linewidth=1.8, linestyle='--', alpha=0.8)
ax.axvline(x=T3, color=COLOR_SEVERE, linewidth=1.8, linestyle='-.', alpha=0.8)

ymax_split = max(kde_safe_v.max(), kde_fail_v.max())
t3_label_x, t3_label_ha = threshold_label_position(T3)
ax.text(T1 + 0.01, ymax_split * 0.92, f'T1={T1:.2f}',
        color=TEXT_SAFE, fontsize=FS_ANNOT, bbox=ANNOT_BOX)
ax.text(T2 + 0.01, ymax_split * 0.82, f'T2={T2:.2f}',
        color=TEXT_FAIL, fontsize=FS_ANNOT, bbox=ANNOT_BOX)
ax.text(t3_label_x, ymax_split * 0.58, f'T3={T3:.2f}',
        color=TEXT_SEVERE, fontsize=FS_ANNOT, ha=t3_label_ha,
        bbox=ANNOT_BOX)

# Zone labels
ax.text((0 + T1)/2,    -ymax_split * 0.06, 'Low',    ha='center', fontsize=FS_ANNOT, color=TEXT_SAFE, bbox=ANNOT_BOX)
ax.text((T1 + T2)/2,   -ymax_split * 0.06, 'Medium', ha='center', fontsize=FS_ANNOT, color=TEXT_MEDIUM, bbox=ANNOT_BOX)
ax.text((T2 + T3)/2,   -ymax_split * 0.06, 'High',   ha='center', fontsize=FS_ANNOT, color=TEXT_FAIL, bbox=ANNOT_BOX)
ax.text((T3 + 1)/2,    -ymax_split * 0.06, 'Severe', ha='center', fontsize=FS_ANNOT, color=TEXT_SEVERE, bbox=ANNOT_BOX)

ax.set_xlabel('Risk Score  (predicted probability of failure)', fontsize=FS_LABEL)
ax.set_ylabel('Density', fontsize=FS_LABEL)
ax.tick_params(labelsize=FS_TICK)
ax.set_xlim(0, 1)
ax.set_ylim(bottom=-ymax_split * 0.10)
ax.legend(loc='upper right',
          fontsize=FS_LEGEND,
          frameon=True, facecolor='white', edgecolor='#BDBDBD',
          framealpha=0.94)
ax.grid(False)
add_bottom_note(fig, 'Split risk score KDE | Safe and confirmed-failure populations shown on the same score axis with T1/T2/T3 boundaries.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_risk_score_split_kde.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_risk_score_split_kde.png'))
plt.close()
print("Saved: fig_risk_score_split_kde")

Saved: fig_risk_score_split_kde


### Plot C — Three-panel view (operational registry)

In [21]:
# ── PLOT C: Three-panel layout — safe / overlap zone / failure ──
# Top panel    : safe pipelines KDE (score 0.0 – 0.5 zoomed in)
# Middle panel : overlap zone (score 0.0 – 1.0 both populations)
# Bottom panel : failure pipelines KDE (score 0.0 – 1.0)
fig, axes = plt.subplots(3, 1, figsize=(16, 21),
                         gridspec_kw={'height_ratios': [1, 1, 1],
                                      'hspace': 0.30},
                         sharex=False)

# Panel 1 — Safe pipeline score distribution (zoomed 0–0.5)
x_zoom = np.linspace(0, 0.5, 300)
axes[0].fill_between(x_zoom, kde_safe(x_zoom), color=COLOR_SAFE, alpha=0.28)
axes[0].plot(x_zoom, kde_safe(x_zoom), color=COLOR_SAFE, linewidth=2.2)
axes[0].axvline(x=T1, color=COLOR_SAFE, linewidth=1.8, linestyle='--')
if T2 <= 0.5: axes[0].axvline(x=T2, color=COLOR_FAIL, linewidth=1.8, linestyle='--')
if T3 <= 0.5: axes[0].axvline(x=T3, color=COLOR_SEVERE, linewidth=1.8, linestyle='-.')
axes[0].set_title('')
axes[0].set_ylabel('Density', fontsize=FS_LABEL)
axes[0].set_xlabel('Risk Score', fontsize=FS_LABEL)
axes[0].set_xlim(0, 0.5)
axes[0].tick_params(labelsize=FS_TICK)
axes[0].grid(False)
y0 = kde_safe(x_zoom).max()
axes[0].text(T1 + 0.005, y0 * 0.88, f'T1={T1:.2f}',
             fontsize=FS_ANNOT, color=TEXT_SAFE, bbox=ANNOT_BOX)
if T2 <= 0.5:
    axes[0].text(T2 + 0.005, y0 * 0.78, f'T2={T2:.2f}',
                 fontsize=FS_ANNOT, color=TEXT_FAIL, bbox=ANNOT_BOX)
if T3 <= 0.5:
    t3_label_x, t3_label_ha = threshold_label_position(T3, x_max=0.5, pad=0.006)
    axes[0].text(t3_label_x, y0 * 0.68, f'T3={T3:.2f}',
                 fontsize=FS_ANNOT, color=TEXT_SEVERE, ha=t3_label_ha,
                 bbox=ANNOT_BOX)

# Panel 2 — Both populations overlaid (full 0–1 range)
axes[1].fill_between(x_grid, kde_safe_v, color=COLOR_SAFE, alpha=0.20,
                     label=f'Safe  (n={len(scores_safe):,})')
axes[1].fill_between(x_grid, kde_fail_v, color=COLOR_FAIL, alpha=0.22,
                     label=f'Failures  (n={len(scores_fail)})')
axes[1].plot(x_grid, kde_safe_v, color=COLOR_SAFE, linewidth=1.8)
axes[1].plot(x_grid, kde_fail_v, color=COLOR_FAIL, linewidth=1.8)
axes[1].axvline(x=T1, color=COLOR_SAFE, linewidth=1.8, linestyle='--')
axes[1].axvline(x=T2, color=COLOR_FAIL, linewidth=1.8, linestyle='--')
axes[1].axvline(x=T3, color=COLOR_SEVERE, linewidth=1.8, linestyle='-.')
axes[1].set_title('')
axes[1].set_ylabel('Density', fontsize=FS_LABEL)
axes[1].set_xlabel('Risk Score', fontsize=FS_LABEL)
axes[1].set_xlim(0, 1)
axes[1].tick_params(labelsize=FS_TICK)
axes[1].legend(loc='upper right', fontsize=FS_LEGEND,
               frameon=True, facecolor='white', edgecolor='#BDBDBD',
               framealpha=0.94)
axes[1].grid(False)

# Zone shading on panel 2 — 4 zones
y1max = max(kde_safe_v.max(), kde_fail_v.max())
axes[1].axvspan(0,   T1, alpha=0.06, color=COLOR_SAFE)
axes[1].axvspan(T1,  T2, alpha=0.06, color=COLOR_MEDIUM)
axes[1].axvspan(T2,  T3, alpha=0.08, color=COLOR_FAIL)
axes[1].axvspan(T3,  1,  alpha=0.15, color=COLOR_SEVERE)
axes[1].text(T1 + 0.01, y1max * 0.92, f'T1={T1:.2f}',
             fontsize=FS_ANNOT, color=TEXT_SAFE, bbox=ANNOT_BOX)
axes[1].text(T2 + 0.01, y1max * 0.82, f'T2={T2:.2f}',
             fontsize=FS_ANNOT, color=TEXT_FAIL, bbox=ANNOT_BOX)
t3_label_x, t3_label_ha = threshold_label_position(T3)
axes[1].text(t3_label_x, y1max * 0.58, f'T3={T3:.2f}',
             fontsize=FS_ANNOT, color=TEXT_SEVERE, ha=t3_label_ha,
             bbox=ANNOT_BOX)
axes[1].text((0 + T1)/2,   y1max * 0.15, 'Low',    ha='center', fontsize=FS_ANNOT, color=TEXT_SAFE, bbox=ANNOT_BOX)
axes[1].text((T1 + T2)/2,  y1max * 0.15, 'Medium', ha='center', fontsize=FS_ANNOT, color=TEXT_MEDIUM, bbox=ANNOT_BOX)
axes[1].text((T2 + T3)/2,  y1max * 0.15, 'High',   ha='center', fontsize=FS_ANNOT, color=TEXT_FAIL, bbox=ANNOT_BOX)
axes[1].text((T3 + 1)/2,   y1max * 0.15, 'Severe', ha='center', fontsize=FS_ANNOT, color=TEXT_SEVERE, bbox=ANNOT_BOX)

# Panel 3 — Failure score distribution (full 0–1)
axes[2].fill_between(x_grid, kde_fail_v, color=COLOR_FAIL, alpha=0.28)
axes[2].plot(x_grid, kde_fail_v, color=COLOR_FAIL, linewidth=2.2)
axes[2].axvline(x=T1, color=COLOR_SAFE, linewidth=1.8, linestyle='--')
axes[2].axvline(x=T2, color=COLOR_FAIL, linewidth=1.8, linestyle='--')
axes[2].axvline(x=T3, color=COLOR_SEVERE, linewidth=1.8, linestyle='-.')
# Rug plot
axes[2].plot(scores_fail,
             np.zeros_like(scores_fail) - kde_fail_v.max() * 0.04,
             '|', color=COLOR_FAIL, alpha=0.5, markersize=10)
axes[2].set_title('')
axes[2].set_xlabel('Risk Score  (predicted probability of failure)', fontsize=FS_LABEL)
axes[2].set_ylabel('Density', fontsize=FS_LABEL)
axes[2].set_xlim(0, 1)
axes[2].tick_params(labelsize=FS_TICK)
axes[2].grid(False)
y2 = kde_fail_v.max()
axes[2].text(T1 + 0.01, y2 * 0.88, f'T1={T1:.2f}',
             fontsize=FS_ANNOT, color=TEXT_SAFE, bbox=ANNOT_BOX)
axes[2].text(T2 + 0.01, y2 * 0.78, f'T2={T2:.2f}',
             fontsize=FS_ANNOT, color=TEXT_FAIL, bbox=ANNOT_BOX)
t3_label_x, t3_label_ha = threshold_label_position(T3)
axes[2].text(t3_label_x, y2 * 0.68, f'T3={T3:.2f}',
             fontsize=FS_ANNOT, color=TEXT_SEVERE, ha=t3_label_ha,
             bbox=ANNOT_BOX)
pct_severe = (scores_fail >= T3).mean() * 100
pct_high   = ((scores_fail >= T2) & (scores_fail < T3)).mean() * 100
pct_mid    = ((scores_fail >= T1) & (scores_fail < T2)).mean() * 100
pct_low    = (scores_fail < T1).mean() * 100
axes[2].text(0.55, y2 * 0.55,
             f'{pct_severe:.0f}% of failures → Severe\n'
             f'{pct_high:.0f}% of failures → High\n'
             f'{pct_mid:.0f}% of failures → Medium\n'
             f'{pct_low:.0f}% of failures → Low',
             fontsize=FS_ANNOT, color=TEXT_SEVERE,
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                       edgecolor=TEXT_SEVERE, alpha=0.92))

add_bottom_note(
    fig,
    'Three-panel risk score distribution | Panel 1: safe pipelines zoomed 0.0-0.5; '
    'Panel 2: safe and failure populations overlaid; Panel 3: confirmed failures. '
    'T1/T2/T3 mark class boundaries.'
)
plt.tight_layout(rect=[0, 0.055, 1, 1], h_pad=1.0)
plt.savefig(os.path.join(PLOTS_DIR, 'fig_risk_score_3panel.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(PLOTS_DIR, 'fig_risk_score_3panel.png'), bbox_inches='tight')
plt.close()
print("Saved: fig_risk_score_3panel")

Saved: fig_risk_score_3panel


## Step 16 — Four-class assignment from the score (OPERATIONAL registry)

In [22]:
# =============================================================================
# STEP 14 — FOUR-CLASS ASSIGNMENT FROM SCORE
# Low    : score < T1  (0.10) — consistent with safe population
# Medium : T1 ≤ score < T2 (0.30) — elevated risk, monitor
# High   : T2 ≤ score < T3 (PR-optimal) — clear failure signal, inspect
# Severe : score ≥ T3 — model is most confident, immediate action
# =============================================================================
print("\n" + "="*70)
print("STEP 14 — FOUR-CLASS ASSIGNMENT FROM SCORE  (OPERATIONAL registry)")
print(f"  T1={T1:.3f} (Low|Medium)  T2={T2:.3f} (Medium|High)  T3={T3:.3f} (High|Severe)")
print("="*70)

def assign_4class(score):
    if   score >= T3: return 'Severe'
    elif score >= T2: return 'High'
    elif score >= T1: return 'Medium'
    else:             return 'Low'

df['risk_class'] = df['risk_score'].apply(assign_4class)

vc = df['risk_class'].value_counts()
print(f"\nFour-class distribution:")
for cls in ORDERED_4:
    n = vc.get(cls, 0)
    print(f"  {cls:8s}: {n:6d}  ({n/len(df)*100:.1f}%)")

print(f"\nConfirmed failures (binary_label=1) per assigned class:")
fail_dist = df[df['binary_label']==1]['risk_class'].value_counts()
for cls in ORDERED_4:
    n = fail_dist.get(cls, 0)
    pct = n / df['binary_label'].sum() * 100
    print(f"  {cls:8s}: {n:5d}  ({pct:.1f}% of all failures)")

print(f"\nMean risk score by assigned class:")
print(df.groupby('risk_class')['risk_score'].mean().round(4).to_string())

# Four-class distribution plot
fig, ax = plt.subplots(figsize=(10, 6))
counts = [vc.get(c, 0) for c in ORDERED_4]
bars = ax.bar(ORDERED_4, counts,
              color=[RISK_COLORS[c] for c in ORDERED_4],
              edgecolor='black', linewidth=1.2, width=0.55)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
            f'{cnt:,}\n({cnt/len(df)*100:.1f}%)',
            ha='center', va='bottom', fontsize=FS_NUMBER)
ax.set_xlabel('Risk Class', fontsize=FS_LABEL)
ax.set_ylabel('Pipeline Count', fontsize=FS_LABEL)
ax.set_title('')
ax.tick_params(labelsize=FS_TICK)
ax.set_ylim(0, max(counts) * 1.25)
ax.grid(False)
add_bottom_note(fig, f'Four-class risk distribution | T1={T1:.2f}; T2={T2:.2f}; T3={T3:.2f}.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_4class_distribution.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_4class_distribution.png'))
plt.close()
print("Saved: fig_4class_distribution")

# =============================================================================


STEP 14 — FOUR-CLASS ASSIGNMENT FROM SCORE  (OPERATIONAL registry)
  T1=0.050 (Low|Medium)  T2=0.300 (Medium|High)  T3=0.872 (High|Severe)

Four-class distribution:
  Severe  :    202  (1.3%)
  High    :    363  (2.4%)
  Medium  :   1737  (11.3%)
  Low     :  13029  (85.0%)

Confirmed failures (binary_label=1) per assigned class:
  Severe  :   202  (33.2% of all failures)
  High    :   286  (47.0% of all failures)
  Medium  :    77  (12.7% of all failures)
  Low     :    43  (7.1% of all failures)

Mean risk score by assigned class:
risk_class
High      0.5849
Low       0.0070
Medium    0.1074
Severe    0.9725


Saved: fig_4class_distribution


## Step 17 — Threshold sensitivity analysis (OPERATIONAL registry)

This is the analysis that, in the original script, was silently computed on the full dataset
while sitting next to "hold-out test set" language in the paper — the exact discrepancy
reviewers caught. Step 18 below produces the honest, test-only counterpart to this table.

In [23]:
# STEP 15 — THRESHOLD SENSITIVITY ANALYSIS  (OPERATIONAL registry, all splits)
# How precision / recall / flagged pipelines change as T2 moves, computed on
# the full 15,331-pipeline registry (train+val+test combined). This describes
# the deployed tool's behaviour across the whole population, NOT a
# generalisation estimate — see Step 18 for the honest test-only version of
# this same sweep, which is what should be cited as the model's true
# held-out performance.
# =============================================================================
print("\n" + "="*70)
print("STEP 15 — THRESHOLD SENSITIVITY ANALYSIS  (OPERATIONAL registry — see Step 18 for honest test-only version)")
print("="*70)

thresholds = np.arange(0.0, 0.996, 0.01)
precs, recs, f1s, flagged = [], [], [], []
true_high = df['binary_label'] == 1

for t in thresholds:
    ph = df['risk_score'] >= t
    tp = (ph & true_high).sum()
    fp = (ph & ~true_high).sum()
    fn = (~ph & true_high).sum()
    p  = tp/(tp+fp) if (tp+fp) else 0
    r  = tp/(tp+fn) if (tp+fn) else 0
    precs.append(p)
    recs.append(r)
    f1s.append(2*p*r/(p+r) if (p+r) else 0)
    flagged.append(ph.sum())

fig, ax1 = plt.subplots(figsize=(16, 8.5))
ax2 = ax1.twinx()

l1, = ax1.plot(thresholds, recs,    color=COLOR_FAIL, lw=2.2, ls='-',  label='Recall')
l2, = ax1.plot(thresholds, precs,   color=COLOR_MEDIUM, lw=2.2, ls='--', label='Precision')
l3, = ax1.plot(thresholds, f1s,     color=COLOR_SAFE, lw=2.2, ls='-.', label='F1-Score')
l4, = ax2.plot(thresholds, flagged, color=COLOR_NEUTRAL, lw=1.5, ls=':',  label='Pipelines Flagged')

# ── T1 — Low|Medium boundary ─────────────────────────────────────────────
ax1.axvline(x=T1, color=COLOR_SAFE, lw=1.8, ls='--', alpha=0.85)
ax1.annotate(f'T1 = {T1:.2f}\nLow | Medium\n(above safe\npopulation mean)',
             xy=(T1, 0.55), xytext=(T1 + 0.04, 0.60),
             fontsize=FS_ANNOT, color=TEXT_SAFE,
             arrowprops=dict(arrowstyle='->', color=COLOR_SAFE, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor=TEXT_SAFE, alpha=0.92))

# ── T2 — Medium|High boundary — curves approximately cross / F1 peaks ────
ax1.axvline(x=T2, color=COLOR_SUCCESS, lw=1.8, ls='--', alpha=0.85)
ax1.annotate(f'T2 = {T2:.2f}\nMedium | High\n(curves cross,\nF1 peaks here)',
             xy=(T2, 0.85), xytext=(T2 + 0.04, 0.78),
             fontsize=FS_ANNOT, color=TEXT_SUCCESS,
             arrowprops=dict(arrowstyle='->', color=COLOR_SUCCESS, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor=TEXT_SUCCESS, alpha=0.92))

# ── T3 — High|Severe boundary — recall stabilises, precision near 1.0 ───
ax1.axvline(x=T3, color=COLOR_SEVERE, lw=1.8, ls='-.', alpha=0.85)
ax1.annotate(f'T3 = {T3:.2f}\nHigh | Severe\n(recall stabilises,\nprecision → 1.0)',
             xy=(T3, 0.40), xytext=(T3 - 0.22, 0.28),
             fontsize=FS_ANNOT, color=TEXT_SEVERE,
             arrowprops=dict(arrowstyle='->', color=COLOR_SEVERE, lw=1.5),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                       edgecolor=TEXT_SEVERE, alpha=0.92))

# ── Zone shading ─────────────────────────────────────────────────────────
ax1.axvspan(0.005, T1, alpha=0.05, color=COLOR_SAFE, label='_nolegend_')
ax1.axvspan(T1,  T2, alpha=0.05, color=COLOR_MEDIUM,  label='_nolegend_')
ax1.axvspan(T2,  T3, alpha=0.05, color=COLOR_FAIL,  label='_nolegend_')
ax1.axvspan(T3,  0.995,alpha=0.10,color=COLOR_SEVERE,  label='_nolegend_')

# Zone labels along bottom
ax1.text((0.005+T1)/2, 0.02, 'Low',    ha='center', fontsize=FS_ANNOT,
         color=TEXT_SAFE, bbox=ANNOT_BOX)
ax1.text((T1+T2)/2,   0.02, 'Medium', ha='center', fontsize=FS_ANNOT,
         color=TEXT_MEDIUM, bbox=ANNOT_BOX)
ax1.text((T2+T3)/2,   0.02, 'High',   ha='center', fontsize=FS_ANNOT,
         color=TEXT_FAIL, bbox=ANNOT_BOX)
ax1.text((T3+0.995)/2, 0.02, 'Severe', ha='center', fontsize=FS_ANNOT,
         color=TEXT_SEVERE, bbox=ANNOT_BOX)

ax1.set_xlabel('Decision Threshold  (score boundary between classes)', fontsize=FS_LABEL)
ax1.set_ylabel('Precision / Recall / F1', fontsize=FS_LABEL)
ax2.set_ylabel('Total Pipelines Flagged', color=COLOR_NEUTRAL, fontsize=FS_LABEL)
ax1.tick_params(labelsize=FS_TICK)
ax2.tick_params(axis='y', labelcolor=COLOR_NEUTRAL, labelsize=FS_TICK)
ax1.set_xlim(0.005, 0.995)
ax1.set_ylim(0, 1.05)
ax1.set_title('')
ax1.legend([l1,l2,l3,l4], [l.get_label() for l in [l1,l2,l3,l4]],
           loc='upper center', bbox_to_anchor=(0.5, -0.10),
           fontsize=FS_LEGEND, ncol=4,
           frameon=True, facecolor='white', edgecolor='#BDBDBD',
           framealpha=0.94)
ax1.grid(False)
add_bottom_note(fig, 'Threshold sensitivity analysis | T2 is selected where curves cross and F1 peaks; T3 marks the severe boundary where recall stabilises.', y=0.006)
plt.tight_layout(rect=[0, 0.16, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_threshold_sensitivity.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_threshold_sensitivity.png'))
plt.close()
print("Saved: fig_threshold_sensitivity")

print("\nThreshold table (T2 options) — OPERATIONAL registry, all splits combined:")
print(f"  {'T2':>6}  {'Caught':>7}  {'FAlarm':>7}  {'Missed':>7}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    ph = df['risk_score'] >= t
    tp = (ph & true_high).sum()
    fp = (ph & ~true_high).sum()
    fn = (~ph & true_high).sum()
    p  = tp/(tp+fp) if (tp+fp) else 0
    r  = tp/(tp+fn) if (tp+fn) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    print(f"  {t:>6.2f}  {tp:>7d}  {fp:>7d}  {fn:>7d}  {p:>6.2f}  {r:>6.2f}  {f1:>6.2f}")


STEP 15 — THRESHOLD SENSITIVITY ANALYSIS  (OPERATIONAL registry — see Step 18 for honest test-only version)


Saved: fig_threshold_sensitivity

Threshold table (T2 options) — OPERATIONAL registry, all splits combined:
      T2   Caught   FAlarm   Missed    Prec     Rec      F1
    0.20      514      209       94    0.71    0.85    0.77
    0.30      488       77      120    0.86    0.80    0.83
    0.40      456       36      152    0.93    0.75    0.83
    0.50      409       18      199    0.96    0.67    0.79
    0.60      364       14      244    0.96    0.60    0.74
    0.70      317        5      291    0.98    0.52    0.68
    0.80      248        2      360    0.99    0.41    0.58


## Step 17b — Final ranked output CSV (OPERATIONAL registry, includes a `split` column)

Each pipeline now records whether it was in `train`, `val`, or `test`, so results can be independently verified or filtered to test-only by a reader.

In [24]:
# =============================================================================
# STEP 16 — FINAL RANKED OUTPUT CSV
# =============================================================================
print("\n" + "="*70)
print("STEP 16 — FINAL RANKED OUTPUT  (OPERATIONAL registry, includes split column)")
print("="*70)

out_cols = ['unique_id', 'line_age_yr', 'diameter_in', 'max_operating_pressure',
            'material', 'fluid', 'elevation', 'length_ft',
            'risk_score', 'risk_class',
            'risk', 'failure_count', 'binary_label', 'split']
out_df = df[[c for c in out_cols if c in df.columns]].sort_values(
    'risk_score', ascending=False)

out_pred = os.path.join(RESULTS_DIR, 'pipeline_risk_predictions.csv')
out_df.to_csv(out_pred, index=False)
print(f"Saved: {out_pred}")

print(f"\nTop 20 highest-risk pipelines:")
print(out_df.head(20)[['unique_id','line_age_yr','diameter_in',
                        'max_operating_pressure','material','fluid',
                        'risk_score','risk_class',
                        'risk','failure_count']].to_string(index=False))


STEP 16 — FINAL RANKED OUTPUT  (OPERATIONAL registry, includes split column)
Saved: Results_Honest/pipeline_risk_predictions.csv

Top 20 highest-risk pipelines:
 unique_id  line_age_yr  diameter_in  max_operating_pressure     material          fluid  risk_score risk_class  risk  failure_count
    147490    18.844627        4.000                   700.0        Steel Produced Water         1.0     Severe   1.0              1
     47115    31.559206        2.450                   150.0 Carbon Steel     Multiphase         1.0     Severe   1.0              1
    147967    17.114305        6.000                   584.4        Steel Produced Water         1.0     Severe   1.0              1
     10954    31.931554        2.900                   150.0 Carbon Steel     Multiphase         1.0     Severe   1.0              1
    147413    18.844627        4.000                   700.0        Steel Produced Water         1.0     Severe   1.0              1
      1432    41.637235        3.000    

## Step 17c — Save trained artifacts

In [25]:
# =============================================================================
# STEP 17 — SAVE MODELS + THRESHOLDS
# =============================================================================
print("\n" + "="*70)
print("STEP 17 — SAVING MODELS + THRESHOLDS")
print("="*70)

joblib.dump(preprocessor, os.path.join(MODELS_DIR, 'preprocessor.pkl'))
joblib.dump(scoring_model, os.path.join(MODELS_DIR, 'best_model_calibrated.pkl'))
joblib.dump({'T1': T1, 'T2': T2, 'T3': T3,
             'best_model': best_name,
             'score_mean': score_mean, 'score_std': score_std,
             'calibrated': results[best_name].get('calibrated', False)},
            os.path.join(MODELS_DIR, 'thresholds.pkl'))

for name, res in results.items():
    fname = name.lower().replace(' ', '_')
    joblib.dump(res['model'], os.path.join(MODELS_DIR, f'model_{fname}.pkl'))
    print(f"Saved: model_{fname}.pkl")

print(f"\nSaved: preprocessor.pkl")
print(f"Saved: thresholds.pkl  (T1={T1:.4f}, T2={T2:.4f})")


STEP 17 — SAVING MODELS + THRESHOLDS
Saved: model_logistic_regression.pkl
Saved: model_random_forest.pkl
Saved: model_svm.pkl
Saved: model_xgboost.pkl
Saved: model_stacking_ensemble.pkl

Saved: preprocessor.pkl
Saved: thresholds.pkl  (T1=0.0500, T2=0.3000)


## Step 18 — Honest held-out evaluation (TEST set, touched exactly once)

Everything above — model selection, threshold derivation, the calibration decision — was made
using only train and validation. `X_test`/`y_test` (defined in Step 4) have not been referenced
until this cell. These are the numbers that belong in the paper's results tables, including the
Brier score reliability diagnostic.

In [26]:
# =============================================================================
# STEP 18 — HONEST HELD-OUT EVALUATION  (TEST SET — touched exactly once)
# Everything above (model selection in Step 8-9, threshold derivation in
# Step 10/13, and the calibration decision in Step 14) was made using only
# TRAIN and VALIDATION. X_test/y_test, defined in Step 4, have not been
# referenced since. This is the ONLY place in the script where the test set
# is used — to report the model's true generalisation performance under the
# thresholds already locked in above. These are the numbers that belong in
# the paper's results tables, NOT the Step 15 operational-registry sweep.
# =============================================================================
print("\n" + "="*70)
print("STEP 18 — HONEST HELD-OUT EVALUATION  (TEST set, touched once)")
print("="*70)

test_scores = scoring_model.predict_proba(X_test_proc)[:, 1]
y_test_pred_default = (test_scores >= 0.5).astype(int)

test_acc     = accuracy_score(y_test, y_test_pred_default)
test_bal_acc = balanced_accuracy_score(y_test, y_test_pred_default)
test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc  = average_precision_score(y_test, test_scores)
test_f1      = f1_score(y_test, y_test_pred_default, pos_label=1)
test_brier   = brier_score_loss(y_test, test_scores)

print(f"Test set size          : {len(y_test)}  ({int(y_test.sum())} confirmed failures)")
print(f"PR-AUC                 : {test_pr_auc:.4f}")
print(f"ROC-AUC                : {test_roc_auc:.4f}")
print(f"Balanced Accuracy      : {test_bal_acc:.4f}")
print(f"F1 (default thr=0.50)  : {test_f1:.4f}")
print(f"Brier score             : {test_brier:.4f}  (reliability diagnostic — lower is better)")
print(classification_report(y_test, y_test_pred_default, target_names=['Safe', 'Failed']))

# Apply the LOCKED T1/T2/T3 thresholds (derived from train/val only, never
# from this test set) to the test set's own scores.
test_df = pd.DataFrame({'risk_score': test_scores, 'binary_label': y_test.values})
test_df['risk_class'] = test_df['risk_score'].apply(assign_4class)

print(f"\nFour-class distribution on TEST set (n={len(test_df)}):")
test_vc = test_df['risk_class'].value_counts()
for cls in ORDERED_4:
    n = test_vc.get(cls, 0)
    print(f"  {cls:8s}: {n:4d}  ({n/len(test_df)*100:.1f}%)")

n_test_fail = int(y_test.sum())
test_fail_dist = test_df[test_df['binary_label']==1]['risk_class'].value_counts()
print(f"\nConfirmed TEST-set failures per assigned class (honest — never used in training):")
for cls in ORDERED_4:
    n = test_fail_dist.get(cls, 0)
    pct = n / n_test_fail * 100 if n_test_fail else 0
    print(f"  {cls:8s}: {n:4d}  ({pct:.1f}% of test-set failures)")

pct_high_severe_test = int(test_fail_dist.get('High', 0) + test_fail_dist.get('Severe', 0))
print(f"\nHONEST HEADLINE NUMBER: {pct_high_severe_test}/{n_test_fail} "
      f"({pct_high_severe_test/n_test_fail*100:.1f}%) of TEST-set confirmed failures "
      f"fall in High or Severe.")
print(f"This is the number that belongs in the paper as the model's held-out "
      f"discriminative ability — not the operational-registry figure from Step 15.")

print(f"\nThreshold sweep on TEST set only (honest counterpart to Step 15's operational table):")
print(f"  {'T2':>6}  {'Caught':>7}  {'FAlarm':>7}  {'Missed':>7}  {'Prec':>6}  {'Rec':>6}  {'F1':>6}")
true_high_test = test_df['binary_label'] == 1
honest_sweep_rows = []
for t in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    ph = test_df['risk_score'] >= t
    tp = (ph & true_high_test).sum()
    fp = (ph & ~true_high_test).sum()
    fn = (~ph & true_high_test).sum()
    p  = tp/(tp+fp) if (tp+fp) else 0
    r  = tp/(tp+fn) if (tp+fn) else 0
    f1 = 2*p*r/(p+r) if (p+r) else 0
    honest_sweep_rows.append({'T2': t, 'Caught': int(tp), 'False Alarms': int(fp),
                               'Missed': int(fn), 'Precision': round(p, 3),
                               'Recall': round(r, 3), 'F1': round(f1, 3)})
    print(f"  {t:>6.2f}  {tp:>7d}  {fp:>7d}  {fn:>7d}  {p:>6.2f}  {r:>6.2f}  {f1:>6.2f}")

honest_sweep_df = pd.DataFrame(honest_sweep_rows)
honest_sweep_df.to_csv(os.path.join(RESULTS_DIR, 'threshold_sensitivity_honest_test.csv'), index=False)
print(f"\nSaved: threshold_sensitivity_honest_test.csv")

test_summary_df = pd.DataFrame([{
    'Model': best_name, 'Split': f'TEST (honest, n={len(y_test)})',
    'PR-AUC': round(test_pr_auc, 4), 'ROC-AUC': round(test_roc_auc, 4),
    'Balanced Acc': round(test_bal_acc, 4), 'F1 (thr=0.5)': round(test_f1, 4),
    'Brier Score': round(test_brier, 4),
}])
test_summary_df.to_csv(os.path.join(RESULTS_DIR, 'model_performance_summary_honest_test.csv'), index=False)
print(f"Saved: model_performance_summary_honest_test.csv")

# =============================================================================
# TRAINING COMPLETE
# =============================================================================
print("\n" + "="*70)
print("TRAINING COMPLETE")
print("="*70)
print(f"\nBest model (selected by VALIDATION PR-AUC): {best_name}")
print(f"Validation PR-AUC     : {results[best_name]['pr_auc']:.4f}")
print(f"Validation ROC-AUC    : {results[best_name]['roc_auc']:.4f}")
print(f"Validation Bal. Acc.  : {results[best_name]['bal_acc']:.4f}")
print(f"Validation Recall     : {results[best_name]['recall_opt']:.4f}")
print(f"Optimal threshold T2  : {results[best_name]['best_threshold']:.4f}  (derived on VALIDATION)")
print(f"Calibrated            : {results[best_name].get('calibrated', False)}")
print(f"\nHONEST TEST-SET PR-AUC : {test_pr_auc:.4f}  (touched once, Step 18 — cite this in the paper)")
print(f"HONEST TEST-SET ROC-AUC: {test_roc_auc:.4f}")
if best_name in cv_results_summary:
    r = cv_results_summary[best_name]
    print(f"\n5-fold CV (leak-free, SMOTE inside folds):")
    print(f"  CV PR-AUC   : {r['pr_auc'][0]:.4f} ± {r['pr_auc'][1]:.4f}")
    print(f"  CV Recall   : {r['recall'][0]:.4f} ± {r['recall'][1]:.4f}")
    print(f"  CV F1       : {r['f1'][0]:.4f} ± {r['f1'][1]:.4f}")
    print(f"  CV Bal-Acc  : {r['bal_acc'][0]:.4f} ± {r['bal_acc'][1]:.4f}")
print(f"\nClass imbalance strategy:")
print(f"  1. BorderlineSMOTE (sampling_strategy=0.3) — boundary-focused oversampling")
print(f"  2. TomekLinks — removes noisy majority samples near the boundary")
print(f"  3. SMOTE applied INSIDE CV folds — prevents metric inflation from leakage")
print(f"  4. class_weight='balanced' on LR / RF / SVM")
print(f"  5. XGBoost scale_pos_weight={imb_ratio:.1f}")
print(f"  6. Sample weights from num_points (normalised, not shown in charts)")
print(f"  7. Threshold from PR curve (not default 0.5)")
print(f"  8. Probability calibration (isotonic regression)")
print(f"  9. PR-AUC as primary metric")
print(f"\nScore thresholds:")
print(f"  T1 = {T1:.4f}  →  Low    | Medium boundary  (below safe population)")
print(f"  T2 = {T2:.4f}  →  Medium | High   boundary  (≥30% failure probability)")
print(f"  T3 = {T3:.4f}  →  High   | Severe boundary  (PR-optimal, immediate action)")
print(f"\nTo change classification without retraining:")
print(f"  Set T1_OVERRIDE and T2_OVERRIDE at the top of this script")
print(f"  and re-run from STEP 13 onward only.")
print(f"\nKey output: {out_pred}")
print(f"  'risk_score' = continuous failure probability  [0-1]")
print(f"  'risk_class' = Low / Medium / High  (from score thresholds)")


STEP 18 — HONEST HELD-OUT EVALUATION  (TEST set, touched once)
Test set size          : 3067  (122 confirmed failures)
PR-AUC                 : 0.6349
ROC-AUC                : 0.9411
Balanced Accuracy      : 0.6914
F1 (default thr=0.50)  : 0.5341
Brier score             : 0.0221  (reliability diagnostic — lower is better)
              precision    recall  f1-score   support

        Safe       0.98      1.00      0.99      2945
      Failed       0.87      0.39      0.53       122

    accuracy                           0.97      3067
   macro avg       0.92      0.69      0.76      3067
weighted avg       0.97      0.97      0.97      3067


Four-class distribution on TEST set (n=3067):
  Severe  :   29  (0.9%)
  High    :   54  (1.8%)
  Medium  :  382  (12.5%)
  Low     : 2602  (84.8%)

Confirmed TEST-set failures per assigned class (honest — never used in training):
  Severe  :   29  (23.8% of test-set failures)
  High    :   31  (25.4% of test-set failures)
  Medium  :   39  (32.

---
# Follow-up Analysis — XGBoost Feature Importance, SHAP, and an Imbalance-Technique Ablation Study

Added in response to SPE Journal reviewer comments asking for feature importance from the
*deployed* model (XGBoost) rather than the Random Forest surrogate used in earlier drafts, a
SHAP-based interpretability check, and an ablation study isolating the contribution of each
imbalance-handling component. Reuses the exact preprocessor and train/validation/test split
from the corrected training run above (same `RAND_SEED=42`) — the test set is not referenced
anywhere in this section.

In [27]:
# =============================================================================
# XGBoost feature importance + SHAP + imbalance-technique ablation study
# Follow-up analysis addressing SPE Journal reviewer comments:
#   - Technical Editor 1, #5: feature importance should match the selected
#     model (XGBoost), not Random Forest.
#   - Technical Editor 2, #6/#7: XGBoost-based SHAP values for interpretability.
#   - Technical Editor 2, #5: ablation study isolating the effect of each
#     imbalance-handling component (class weighting, SMOTE, SMOTE+TomekLinks,
#     sample weights, and the full combined method).
#
# Reuses the exact preprocessor and train/validation/test split from
# train_pipeline_risk_honest_split.py (same RAND_SEED=42) so results are
# directly comparable to the numbers already in the manuscript. All new
# metrics below are computed on VALIDATION ONLY — the test set defined in
# that script is not touched here.
# =============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

warnings.filterwarnings('ignore')
RAND_SEED = 42
np.random.seed(RAND_SEED)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score,
                              balanced_accuracy_score, recall_score)
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.under_sampling import TomekLinks
from xgboost import XGBClassifier

DATA_PATH   = "Data/pipeline_level_dataset__2024.csv"
MODELS_DIR  = "Results_Honest/Models"
RESULTS_DIR = "Results_Honest"
PLOTS_DIR   = "Results_Honest/Plots"

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 15, 'axes.linewidth': 1.0,
    'axes.labelsize': 16, 'axes.titlesize': 18, 'xtick.labelsize': 14,
    'ytick.labelsize': 14, 'legend.frameon': False, 'legend.fontsize': 13,
    'figure.dpi': 300, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
    'savefig.facecolor': 'white', 'pdf.fonttype': 42, 'ps.fonttype': 42
})
COLOR_SAFE, COLOR_FAIL, COLOR_SUCCESS = '#0072B2', '#D55E00', '#009E73'
FS_LABEL, FS_TICK, FS_NUMBER, FS_NOTE = 16, 14, 16, 12

def add_bottom_note(fig, text, y=0.012):
    fig.text(0.5, y, text, ha='center', va='bottom', fontsize=FS_NOTE, color='#2F2F2F')

In [28]:
# =============================================================================
# Recreate the exact same train/validation/test split
# =============================================================================
df = pd.read_csv(DATA_PATH)
df['binary_label'] = (df['failure_count'] >= 1).astype(int)

NUMERIC_FEATURES = ['line_age_yr', 'max_operating_pressure', 'diameter_in',
                     'elevation', 'length_ft', 'num_points']
CATEGORICAL_FEATURES = ['status', 'flowline_action', 'location_type', 'fluid', 'material']

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df['binary_label'].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=RAND_SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RAND_SEED, stratify=y_temp)

print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)} (untouched here)")

preprocessor = joblib.load(os.path.join(MODELS_DIR, 'preprocessor.pkl'))
X_train_proc = preprocessor.transform(X_train)
X_val_proc   = preprocessor.transform(X_val)

imb_ratio = (y_train == 0).sum() / (y_train == 1).sum()
num_pts_train = X_train['num_points'].fillna(X_train['num_points'].median())
sample_weights = (num_pts_train / num_pts_train.mean()).values

ohe_feats = list(preprocessor.named_transformers_['cat']
                  .named_steps['encoder'].get_feature_names_out(CATEGORICAL_FEATURES))
all_feats = NUMERIC_FEATURES + ohe_feats

label_map = {
    'line_age_yr': 'Age (years)', 'max_operating_pressure': 'MAOP (psi)',
    'diameter_in': 'Diameter (inches)', 'elevation': 'Elevation (m)',
    'length_ft': 'Length (ft)', 'status': 'Status',
    'flowline_action': 'Regulatory Action', 'location_type': 'Connected Facility Type',
    'fluid': 'Fluid Type', 'material': 'Pipe Material',
}

Train: 9198  Val: 3066  Test: 3067 (untouched here)


## Part 1 — XGBoost native (gain-based) feature importance

Replaces the Random Forest feature-importance chart used in earlier drafts, so the reported rankings match the model actually deployed for risk scoring.

In [29]:
# =============================================================================
# PART 1 — XGBoost native feature importance (same style as the RF chart)
# =============================================================================
print("\n" + "=" * 70)
print("PART 1 — XGBOOST FEATURE IMPORTANCE")
print("=" * 70)

xgb_model = joblib.load(os.path.join(MODELS_DIR, 'model_xgboost.pkl'))
imps = xgb_model.feature_importances_

imp_dict = {}
for feat in NUMERIC_FEATURES:
    imp_dict[feat] = imps[all_feats.index(feat)]
for cat in CATEGORICAL_FEATURES:
    idxs = [i for i, f in enumerate(all_feats) if f.startswith(cat)]
    imp_dict[cat] = sum(imps[i] for i in idxs)

imp_df = pd.DataFrame([
    {'Feature': label_map.get(k, k), 'Importance': v}
    for k, v in imp_dict.items() if k != 'num_points'
]).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6.5))
bars = ax.barh(imp_df['Feature'], imp_df['Importance'], color=COLOR_FAIL,
               edgecolor='black', linewidth=0.8)
max_imp = imp_df['Importance'].max()
for bar, val in zip(bars, imp_df['Importance']):
    ax.text(val + max_imp * 0.025, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=FS_NUMBER)
ax.set_xlabel('Feature Importance (XGBoost — gain-based, binary model)', fontsize=FS_LABEL)
ax.tick_params(labelsize=FS_TICK)
ax.set_xlim(0, max_imp * 1.22)
ax.grid(False)
add_bottom_note(fig, 'XGBoost gain-based feature importance (the deployed model); num_points excluded as a GIS segment count, not a physical property.')
plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_feature_importance_xgboost.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_feature_importance_xgboost.png'))
plt.close()

print(imp_df.sort_values('Importance', ascending=False).to_string(index=False))
imp_df.sort_values('Importance', ascending=False).to_csv(
    os.path.join(RESULTS_DIR, 'feature_importance_xgboost.csv'), index=False)
print("Saved: fig_feature_importance_xgboost, feature_importance_xgboost.csv")

# =============================================================================


PART 1 — XGBOOST FEATURE IMPORTANCE


                Feature  Importance
Connected Facility Type    0.544739
          Pipe Material    0.145869
             Fluid Type    0.108649
      Regulatory Action    0.091270
             MAOP (psi)    0.029320
                 Status    0.020135
          Elevation (m)    0.013191
            Age (years)    0.010994
      Diameter (inches)    0.009542
            Length (ft)    0.008056
Saved: fig_feature_importance_xgboost, feature_importance_xgboost.csv


## Part 2 — SHAP analysis

Gain-based importance can overweight features used frequently as early tree splits, regardless
of their actual effect size. SHAP values give a theoretically grounded, consistency-preserving
alternative — computed here on the validation set, matching the paper's convention of excluding
`num_points` (a GIS segment count, not a physical property) from interpretation.

In [30]:
# PART 2 — SHAP summary plot (TreeExplainer on the XGBoost model)
# =============================================================================
print("\n" + "=" * 70)
print("PART 2 — SHAP ANALYSIS")
print("=" * 70)

explainer = shap.TreeExplainer(xgb_model)
# Use validation set for SHAP (never trained on, still large enough: n=3066)
shap_values = explainer.shap_values(X_val_proc)

# Exclude num_points, consistent with the gain-importance chart above — it is a
# GIS segment count, not an independent physical property of the pipeline.
npt_idx = all_feats.index('num_points')
keep_idx = [i for i in range(len(all_feats)) if i != npt_idx]
shap_values_kept = shap_values[:, keep_idx]
feats_kept = [all_feats[i] for i in keep_idx]

def readable_name(f):
    if f in label_map:
        return label_map[f]
    for cat in CATEGORICAL_FEATURES:
        if f.startswith(cat + '_'):
            return f'{label_map[cat]}: {f[len(cat) + 1:]}'
    return f

readable_feats = [readable_name(f) for f in feats_kept]

plt.rcParams.update({'font.size': 20})
plt.figure()
shap.summary_plot(shap_values_kept, X_val_proc[:, keep_idx], feature_names=readable_feats,
                   show=False, max_display=15, plot_size=(14, 10))
fig = plt.gcf()
ax = fig.axes[0]
ax.tick_params(labelsize=19)
ax.xaxis.label.set_size(20)
# enlarge the beeswarm points (SHAP's default markers are quite small)
for coll in ax.collections:
    sizes = coll.get_sizes()
    if len(sizes) > 0:
        coll.set_sizes(sizes * 2.6)
if len(fig.axes) > 1:
    cax = fig.axes[1]
    cax.tick_params(labelsize=16)
    cax.set_ylabel(cax.get_ylabel(), fontsize=18)
add_bottom_note(fig, 'SHAP summary (XGBoost, validation set) | num_points excluded, consistent with Figure 7 | Color = feature value, position = impact on predicted failure probability.', y=-0.02)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'fig_shap_summary.png'), bbox_inches='tight', dpi=400)
plt.savefig(os.path.join(PLOTS_DIR, 'fig_shap_summary.pdf'), bbox_inches='tight')
plt.close()
plt.rcParams.update({'font.size': 15})
print("Saved: fig_shap_summary")

# Aggregate one-hot columns back to their parent categorical feature, matching
# the gain-importance chart's aggregation, for an apples-to-apples ranking.
mean_abs_shap = np.abs(shap_values_kept).mean(axis=0)
agg = {}
for f, v in zip(feats_kept, mean_abs_shap):
    parent = f
    for cat in CATEGORICAL_FEATURES:
        if f.startswith(cat + '_'):
            parent = cat
            break
    agg[parent] = agg.get(parent, 0) + v
shap_imp_df = pd.DataFrame([{'Feature': label_map.get(k, k), 'Mean |SHAP|': v}
                            for k, v in agg.items()]).sort_values('Mean |SHAP|', ascending=False)
print(shap_imp_df.to_string(index=False))
shap_imp_df.to_csv(os.path.join(RESULTS_DIR, 'shap_importance_aggregated.csv'), index=False)
print("Saved: shap_importance_aggregated.csv")

# =============================================================================


PART 2 — SHAP ANALYSIS


Saved: fig_shap_summary
                Feature  Mean |SHAP|
Connected Facility Type     1.337282
          Elevation (m)     1.100273
             MAOP (psi)     1.030377
            Length (ft)     0.957301
          Pipe Material     0.625529
      Diameter (inches)     0.441831
             Fluid Type     0.389367
            Age (years)     0.303066
      Regulatory Action     0.209755
                 Status     0.037847
Saved: shap_importance_aggregated.csv


## Part 3 — Ablation study on imbalance-handling techniques

Isolates the contribution of each component in the imbalance-handling stack (Step 6/6b above):
class weighting alone, SMOTE alone, SMOTE+TomekLinks, sample weighting alone, and the full
combined method. Each configuration is a single XGBoost training run, evaluated once on the
validation set — the test set is not used here.

In [31]:
# PART 3 — Ablation study on imbalance-handling techniques
# All 5 configs trained on TRAIN, evaluated on VALIDATION only.
# =============================================================================
print("\n" + "=" * 70)
print("PART 3 — IMBALANCE TECHNIQUE ABLATION STUDY  (XGBoost, validation set)")
print("=" * 70)

def eval_config(model, Xv_proc, yv):
    prob = model.predict_proba(Xv_proc)[:, 1]
    pred = (prob >= 0.5).astype(int)
    return {
        'PR-AUC': average_precision_score(yv, prob),
        'ROC-AUC': roc_auc_score(yv, prob),
        'Recall': recall_score(yv, pred),
        'F1': f1_score(yv, pred),
        'Balanced Acc': balanced_accuracy_score(yv, pred),
    }

results = {}

# --- Config A: class weighting only (scale_pos_weight, no resampling, no sample weights) ---
m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                   colsample_bytree=0.8, scale_pos_weight=imb_ratio, eval_metric='aucpr',
                   random_state=RAND_SEED, n_jobs=-1, verbosity=0)
m.fit(X_train_proc, y_train)
results['Class weighting only'] = eval_config(m, X_val_proc, y_val)
print('Class weighting only        done')

# --- Config B: SMOTE only (BorderlineSMOTE, no Tomek, no scale_pos_weight, no sample weights) ---
bsmote = BorderlineSMOTE(sampling_strategy=0.3, random_state=RAND_SEED, k_neighbors=5)
X_res_b, y_res_b = bsmote.fit_resample(X_train_proc, y_train)
m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                   colsample_bytree=0.8, eval_metric='aucpr',
                   random_state=RAND_SEED, n_jobs=-1, verbosity=0)
m.fit(X_res_b, y_res_b)
results['SMOTE only'] = eval_config(m, X_val_proc, y_val)
print('SMOTE only                  done')

# --- Config C: SMOTE + TomekLinks (no scale_pos_weight, no sample weights) ---
tomek = TomekLinks()
X_res_c, y_res_c = tomek.fit_resample(X_res_b, y_res_b)
m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                   colsample_bytree=0.8, eval_metric='aucpr',
                   random_state=RAND_SEED, n_jobs=-1, verbosity=0)
m.fit(X_res_c, y_res_c)
results['SMOTE + TomekLinks'] = eval_config(m, X_val_proc, y_val)
print('SMOTE + TomekLinks           done')

# --- Config D: sample weights only (num_points-based, no resampling, no class weighting) ---
m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                   colsample_bytree=0.8, eval_metric='aucpr',
                   random_state=RAND_SEED, n_jobs=-1, verbosity=0)
m.fit(X_train_proc, y_train, sample_weight=sample_weights)
results['Sample weights only'] = eval_config(m, X_val_proc, y_val)
print('Sample weights only          done')

# --- Config E: full method (SMOTE+Tomek + scale_pos_weight + sample weights) ---
n_diff = len(X_res_c) - len(X_train_proc)
if n_diff >= 0:
    sw_res = np.concatenate([sample_weights, np.ones(n_diff)])
else:
    sw_res = np.ones(len(X_res_c))
m = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8,
                   colsample_bytree=0.8, scale_pos_weight=imb_ratio, eval_metric='aucpr',
                   random_state=RAND_SEED, n_jobs=-1, verbosity=0)
m.fit(X_res_c, y_res_c, sample_weight=sw_res)
results['Full method (all combined)'] = eval_config(m, X_val_proc, y_val)
print('Full method (all combined)   done')

ablation_df = pd.DataFrame(results).T
ablation_df = ablation_df[['PR-AUC', 'ROC-AUC', 'Recall', 'F1', 'Balanced Acc']].round(4)
print()
print(ablation_df.to_string())
ablation_df.to_csv(os.path.join(RESULTS_DIR, 'ablation_study.csv'))
print("Saved: ablation_study.csv")

# Plot
fig, ax = plt.subplots(figsize=(11, 6.5))
order = ['Class weighting only', 'SMOTE only', 'SMOTE + TomekLinks',
         'Sample weights only', 'Full method (all combined)']
x = np.arange(len(order))
width = 0.25
metrics_to_plot = ['PR-AUC', 'Recall', 'F1']
colors = [COLOR_SAFE, COLOR_FAIL, COLOR_SUCCESS]
for i, (metric, color) in enumerate(zip(metrics_to_plot, colors)):
    vals = [results[cfg][metric] for cfg in order]
    bars = ax.bar(x + (i - 1) * width, vals, width, label=metric, color=color,
                   edgecolor='black', linewidth=0.6)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.015, f'{v:.2f}',
                ha='center', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=20, ha='right', fontsize=12)
ax.set_ylabel('Score (validation set)', fontsize=FS_LABEL)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper left', fontsize=13, frameon=True, facecolor='white', edgecolor='#BDBDBD')
ax.grid(False)
add_bottom_note(fig, 'Ablation study: XGBoost trained under 5 imbalance-handling configurations, evaluated once on the validation set.')
plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.savefig(os.path.join(PLOTS_DIR, 'fig_ablation_study.pdf'))
plt.savefig(os.path.join(PLOTS_DIR, 'fig_ablation_study.png'))
plt.close()
print("Saved: fig_ablation_study")

print("\nDONE.")


PART 3 — IMBALANCE TECHNIQUE ABLATION STUDY  (XGBoost, validation set)


Class weighting only        done


SMOTE only                  done


SMOTE + TomekLinks           done


Sample weights only          done


Full method (all combined)   done

                            PR-AUC  ROC-AUC  Recall      F1  Balanced Acc
Class weighting only        0.6619   0.9424  0.6777  0.5395        0.8217
SMOTE only                  0.7198   0.9483  0.5702  0.6635        0.7821
SMOTE + TomekLinks          0.7089   0.9468  0.5785  0.6604        0.7857
Sample weights only         0.6388   0.9379  0.3967  0.5304        0.6963
Full method (all combined)  0.6778   0.9446  0.7603  0.4589        0.8482
Saved: ablation_study.csv


Saved: fig_ablation_study

DONE.


---
# New Addition — Interactive Pipeline Risk Assessment

Everything above reproduces the corrected training script exactly. This final section is the
new addition: given the specifications of a single pipeline, it runs that pipeline through the
exact artifacts produced above — the same `preprocessor`, the same calibrated `scoring_model`,
the same `T1`/`T2`/`T3` thresholds (all derived from Train/Validation only, never Test) — and
returns:

- a continuous **risk score** (predicted probability of failure, 0-1)
- a **risk class**: `Low` / `Medium` / `High` / `Severe`, using the identical rule as `assign_4class`

There are two ways to use it below: an **interactive form** (sliders / dropdowns + a button,
for a live Jupyter/VS Code kernel), and **direct function calls** you can use from code or loop
over a batch of pipelines.

In [32]:
import ipywidgets as widgets
from IPython.display import display, HTML

RISK_DESCRIPTIONS = {
    'Low':    'Consistent with the safe population. No action needed.',
    'Medium': 'Elevated risk — shares some characteristics with historical failures. Monitor.',
    'High':   'Clear failure signal. Schedule inspection.',
    'Severe': "Model is most confident about failure risk. Do not wait for the next scheduled inspection cycle.",
}
RISK_BADGE_COLOR = {
    'Low': COLOR_SAFE, 'Medium': COLOR_MEDIUM, 'High': COLOR_HIGH, 'Severe': COLOR_SEVERE,
}

# num_points is the GIS 50 m-segment count used during training — not something a user
# describing a pipeline would know offhand. It reconstructs cleanly from length_ft alone:
# across the training data, length_ft / num_points has a median of ~156 ft (vs. 50 m = 164 ft),
# and round(length_ft / FT_PER_SEGMENT) exactly matches the recorded num_points 69% of the time
# and is within +/-1 segment 97% of the time. So it is auto-derived, not asked for.
FT_PER_SEGMENT = 50 * 3.28084  # 50 metres in feet

def estimate_num_points(length_ft: float) -> int:
    return max(1, round(length_ft / FT_PER_SEGMENT))

def assess_pipeline(spec: dict, verbose: bool = True):
    """Score a single pipeline specification with the trained model.

    `spec` must provide 10 of the 11 model features:
    line_age_yr, max_operating_pressure, diameter_in, elevation, length_ft,
    status, flowline_action, location_type, fluid, material.
    `num_points` (GIS 50 m-segment count) is auto-derived from length_ft unless
    explicitly supplied in `spec`.
    Returns (risk_score, risk_class).
    """
    spec = dict(spec)
    spec.setdefault('num_points', estimate_num_points(spec['length_ft']))
    row = pd.DataFrame([{f: spec[f] for f in NUMERIC_FEATURES + CATEGORICAL_FEATURES}])
    row_proc = preprocessor.transform(row)
    score = float(scoring_model.predict_proba(row_proc)[:, 1][0])
    risk_class = assign_4class(score)

    if verbose:
        color = RISK_BADGE_COLOR[risk_class]
        display(HTML(f'''
        <div style="border:2px solid {color}; border-radius:10px; padding:14px 18px;
                    font-family:sans-serif; max-width:560px;">
          <div style="font-size:15px; color:#555;">Predicted risk score</div>
          <div style="font-size:30px; font-weight:700; color:{color};">{score:.3f}
             <span style="font-size:16px; font-weight:600; color:{color};"> — {risk_class.upper()}</span>
          </div>
          <div style="height:10px; background:#eee; border-radius:5px; margin:8px 0 10px 0;">
            <div style="height:10px; width:{max(score,0.01)*100:.1f}%; background:{color}; border-radius:5px;"></div>
          </div>
          <div style="font-size:13px; color:#333;">{RISK_DESCRIPTIONS[risk_class]}</div>
          <div style="font-size:12px; color:#888; margin-top:8px;">
            Thresholds — Low &lt; {T1:.2f} ≤ Medium &lt; {T2:.2f} ≤ High &lt; {T3:.2f} ≤ Severe
          </div>
        </div>
        '''))
    return score, risk_class

## Interactive form

Fill in the specifications of the pipeline you want to check and click **Assess Risk**.
Dropdown options are the categories the model was trained on
(from `Data/pipeline_level_dataset__2024.csv`); numeric ranges reflect the observed data range.

Note: `num_points` (the GIS 50 m-segment count) is **not** an input here — it is auto-derived
from `length_ft` (see `estimate_num_points` above), since it is a digitization artifact rather
than something you'd know about a pipeline directly.

In [33]:
status_opts    = sorted(df['status'].unique().tolist())
action_opts    = sorted(df['flowline_action'].unique().tolist())
location_opts  = sorted(df['location_type'].unique().tolist())
fluid_opts     = sorted(df['fluid'].unique().tolist())
material_opts  = sorted(df['material'].unique().tolist())

w_age      = widgets.FloatSlider(value=15.0, min=0, max=75, step=0.5, description='Age (yr)', style={'description_width': '140px'}, layout=widgets.Layout(width='420px'))
w_maop     = widgets.FloatSlider(value=200.0, min=0, max=3800, step=10, description='MAOP (psi)', style={'description_width': '140px'}, layout=widgets.Layout(width='420px'))
w_diameter = widgets.FloatSlider(value=6.0, min=0.5, max=24, step=0.5, description='Diameter (in)', style={'description_width': '140px'}, layout=widgets.Layout(width='420px'))
w_elev     = widgets.FloatSlider(value=1700.0, min=900, max=2800, step=10, description='Elevation (m)', style={'description_width': '140px'}, layout=widgets.Layout(width='420px'))
w_length   = widgets.FloatText(value=1000.0, description='Length (ft)', style={'description_width': '140px'}, layout=widgets.Layout(width='300px'))

w_status   = widgets.Dropdown(options=status_opts,   value='Active',       description='Status',           style={'description_width': '140px'})
w_action   = widgets.Dropdown(options=action_opts,    value='Registration', description='Regulatory action',style={'description_width': '140px'})
w_location = widgets.Dropdown(options=location_opts,  value='Well Site',    description='Connected facility',style={'description_width': '140px'})
w_fluid    = widgets.Dropdown(options=fluid_opts,     value='Crude Oil',    description='Fluid',            style={'description_width': '140px'})
w_material = widgets.Dropdown(options=material_opts,  value='Carbon Steel', description='Material',         style={'description_width': '140px'})

btn = widgets.Button(description='Assess Risk', button_style='danger', icon='exclamation-triangle')
out = widgets.Output()

def _on_click(b):
    spec = dict(
        line_age_yr=w_age.value, max_operating_pressure=w_maop.value,
        diameter_in=w_diameter.value, elevation=w_elev.value,
        length_ft=w_length.value,
        status=w_status.value, flowline_action=w_action.value,
        location_type=w_location.value, fluid=w_fluid.value, material=w_material.value,
    )
    with out:
        out.clear_output()
        assess_pipeline(spec)

btn.on_click(_on_click)

form = widgets.VBox([
    widgets.HTML('<b>Physical specifications</b>'),
    w_age, w_maop, w_diameter, w_elev, w_length,
    widgets.HTML('<b>Categorical attributes</b>'),
    w_status, w_action, w_location, w_fluid, w_material,
    btn, out,
])
display(form)

## Direct calls (works without a live kernel — e.g. when this notebook is only viewed as static HTML)

Four example pipeline specifications scored directly through `assess_pipeline`, chosen to land
in each of the four risk classes so the full range of outcomes is visible.

In [34]:
example_pipelines = {
    'Low — new HDPE gathering line': dict(
        line_age_yr=3, max_operating_pressure=60, diameter_in=4, elevation=2200,
        length_ft=500, num_points=2, status='Active', flowline_action='Registration',
        location_type='Well Site', fluid='Natural Gas', material='HDPE'),

    'Medium — older HDPE well-site line, high MAOP': dict(
        line_age_yr=45, max_operating_pressure=968, diameter_in=6, elevation=1421,
        length_ft=7300, num_points=45, status='Active', flowline_action='Registration',
        location_type='Well Site', fluid='Produced Water', material='HDPE'),

    'High — mid-age carbon steel line at a production facility': dict(
        line_age_yr=6.5, max_operating_pressure=245, diameter_in=3, elevation=1702,
        length_ft=1346, num_points=10, status='Active', flowline_action='Registration',
        location_type='Production Facilities', fluid='Co2/Produced Water', material='Carbon Steel'),

    'Severe — HDPE well-site line, high MAOP, long and heavily segmented': dict(
        line_age_yr=14.4, max_operating_pressure=968, diameter_in=6, elevation=1421,
        length_ft=7300, num_points=45, status='Active', flowline_action='Registration',
        location_type='Well Site', fluid='Produced Water', material='HDPE'),
}

for label, spec in example_pipelines.items():
    print(f"{'='*70}\n{label}\n{'='*70}")
    assess_pipeline(spec)

Low — new HDPE gathering line


Medium — older HDPE well-site line, high MAOP


High — mid-age carbon steel line at a production facility


Severe — HDPE well-site line, high MAOP, long and heavily segmented
